# PDDL Attack Path Evaluation
Evaluate generated PDDL attack paths: solvability, syntax, and semantic quality.

## 1. Environment Setup (imports, constnats, global variables)

In [26]:
import sys
import os
import re
import json
import time
import random as _rng
try:
    import resource  # Unix-only
except ImportError:
    resource = None
import platform
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from openai import OpenAI
from jinja2 import Environment, FileSystemLoader
from ipywidgets import interact, IntSlider

from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.model_selection import GroupKFold
from statsmodels.stats.inter_rater import fleiss_kappa as _fleiss_kappa, aggregate_raters

from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

from cve2pddlap.core.data_loader import load_few_shot_pool
from cve2pddlap.evaluation import create_ff_checker, create_enhsp_checker
from cve2pddlap.evaluation.problem_pddl_generator import generate_problem
from cve2pddlap.llm_providers.remote.openai_compat import QwenProvider

# Project root (relative — works from notebooks/attack_paths/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

# Data paths and file names
DATASET_PATH = os.path.join(PROJECT_ROOT, 'resources', 'data', 'CVE-PDDL-NNL-ReAP')
BAD_PDDL_DIR = os.path.join(PROJECT_ROOT, 'resources', 'data', 'mutated_bad_PDDL_AP')
TARGET_POOL_FILE = os.path.join(PROJECT_ROOT, 'resources', 'data', 'target_pool.json')
GENERATED_DOMAIN_DIR = os.path.join(PROJECT_ROOT, 'generated_domain')
FIG_DIR = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_intrinsic')
os.makedirs(FIG_DIR, exist_ok=True)
FIG_DIR_EXT = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_extrinsic')
os.makedirs(FIG_DIR_EXT, exist_ok=True)
EVAL_SET_DIR = os.path.join(GENERATED_DOMAIN_DIR, 'eval_set')
PROMPTS_PATH = os.path.join(PROJECT_ROOT, 'resources', 'prompt', 'evaluation')
AP_PATTERN = re.compile(r'^AP\d+$')
DOMAIN_FILE = 'domain.pddl'
PROBLEM_FILE = 'problem.pddl'

# Models (small defaults — replace with preferred models)
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-8B"  # too large for CPU
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"  # Qwen3 embedding model 4B
EMBEDDING_MODEL_NAME_2 = "BAAI/bge-base-en-v1.5"  # fallback: smaller model


LLM_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# Generation parameters
SEED = 42
TEMPERATURE = 0.0
TOP_K = 1


@dataclass(frozen=True)
class EvaluationFlags:
    syntax_check: bool = True
    embedding_intrinsic: bool = True
    embedding_extrinsic: bool = True
    llm_intrinsic: bool = True
    llm_extrinsic: bool = True


eval_flags = EvaluationFlags()

os.environ['MallocStackLogging'] = '0'

# Results output directory
RESULTS_BASE = os.path.join(PROJECT_ROOT, "results", "tests", "reference_set")

def get_device_info():
    """Return device info dict."""
    return {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "python": platform.python_version(),
        "torch_device": "cuda" if torch.cuda.is_available() else "cpu",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    }


## 2. Load data, Prompts, LLM (tokenizer and embedding model)

In [38]:
few_shot_pool = load_few_shot_pool(DATASET_PATH)
print(f'Reference examples: {len(few_shot_pool)}')
for ex in few_shot_pool[:5]:
    print(f'  {ex.key}')
print('  ...')
print(f'Generated domain dir to be evaluated: {os.path.relpath(EVAL_SET_DIR)}')

Reference examples: 55
  CVE-2022-1471 / AP1
  CVE-2022-40149 / AP1
  CVE-2022-40149 / AP2
  CVE-2022-40150 / AP1
  CVE-2022-40150 / AP2
  ...
Generated domain dir to be evaluated: ..\..\generated_domain\eval_set


In [39]:
def model_short_name(model_name):
    """Generate a short readable name from a full model identifier.
         'BAAI/bge-base-en-v1.5' -> 'bge-base-en-v1.5'
         'Qwen/Qwen3-Embedding-4B' -> 'Qwen3-Embedding-4B'
    """
    name = model_name.split("/")[-1]  # strip org prefix
    for suffix in ["-instruct", "-Instruct", "-chat", "-Chat"]:
        if name.endswith(suffix):
            name = name[:-len(suffix)]
            break
    return name


def load_target_pool(target_pool_file):
    """Load CVE descriptions from target_pool.json. Returns {cve_id: description}."""
    with open(target_pool_file, encoding='utf-8') as f:
        pool = json.load(f)
    return {entry['cve_id']: entry['description'] for entry in pool}


def load_dataset(data_path, cve_descriptions):
    """Load all CVEs with their descriptions (from target_pool) and attack path PDDL files."""
    dataset = []
    for cve_dir in sorted(Path(data_path).iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id)
        if description is None:
            continue
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir() or not AP_PATTERN.match(ap_dir.name):
                continue
            domain_file = ap_dir / DOMAIN_FILE
            problem_file = ap_dir / PROBLEM_FILE
            if domain_file.exists() and problem_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                    'problem': problem_file.read_text(encoding='utf-8').strip(),
                })
        dataset.append({
            'cve_id': cve_id,
            'description': description,
            'attack_paths': attack_paths,
        })
    return dataset


def load_bad_dataset(bad_pddl_dir, cve_descriptions):
    """Load mutated bad PDDL domains from mutated_bad_PDDL_AP/.
    Returns list of {cve_id, description, attack_paths: [{ap_id, domain}]}."""
    bad_dataset = []
    bad_dir = Path(bad_pddl_dir)
    if not bad_dir.exists():
        print(f"WARNING: {bad_pddl_dir} not found")
        return bad_dataset
    for cve_dir in sorted(bad_dir.iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id, "")
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir():
                continue
            domain_file = ap_dir / DOMAIN_FILE
            if domain_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                })
        if attack_paths:
            bad_dataset.append({
                'cve_id': cve_id,
                'description': description,
                'attack_paths': attack_paths,
            })
    return bad_dataset

def load_prompts(prompts_path):
    """Load Jinja2 evaluation prompt templates."""
    return Environment(loader=FileSystemLoader(prompts_path))


MODEL_VRAM_GB = {
    # rough fp16/bf16 weight footprint for GPU/CPU decision
    "Qwen3-Embedding-8B": 16.5,
    "Qwen3-Embedding-4B": 8.5,
    "Qwen3-Embedding-0.6B": 1.5,
    "jina-embeddings-v3": 1.5,
    "stella_en_1.5B": 3.5,
    "nomic-embed": 1.0,
    "mxbai-embed-large": 1.5,
    "bge-m3": 1.5,
    "bge-base": 0.5,
    "all-MiniLM": 0.2,
}

def _estimate_vram_gb(model_name):
    for key, gb in MODEL_VRAM_GB.items():
        if key in model_name:
            return gb
    return 2.0  # default safety estimate

def load_embedding_model(model_name, headroom=1.4):
    """Load a SentenceTransformer bi-encoder.
    Pre-check free VRAM: if not enough (estimated * headroom) -> CPU.
    Avoids Windows WDDM silent shared-memory spillover (~10x slower than CPU).
    Always falls back to CPU on OOM.
    """
    TRUST_REMOTE = ["Qwen3-Embedding", "nomic-ai/", "jinaai/jina-embeddings-v3"]
    BF16_ON_CPU = ["Qwen3-Embedding-4B", "Qwen3-Embedding-8B"]
    kwargs = {"trust_remote_code": True} if any(t in model_name for t in TRUST_REMOTE) else {}

    needed = _estimate_vram_gb(model_name) * headroom
    use_gpu = False
    free_gb = 0.0
    if torch.cuda.is_available():
        free_gb = torch.cuda.mem_get_info()[0] / 1e9
        if free_gb >= needed:
            use_gpu = True
        else:
            print(f"  [VRAM CHECK] {model_name}: need ~{needed:.1f} GB but only {free_gb:.1f} GB free -> CPU")
    if not use_gpu:
        cpu_kwargs = dict(kwargs)
        if any(t in model_name for t in BF16_ON_CPU):
            cpu_kwargs["model_kwargs"] = {"torch_dtype": torch.bfloat16}
        model = SentenceTransformer(model_name, device="cpu", **cpu_kwargs)
        print(f"  Loaded {model_name} on CPU{' (bf16)' if 'model_kwargs' in cpu_kwargs else ''}")
        return model
    try:
        model = SentenceTransformer(model_name, **kwargs)
        print(f"  Loaded {model_name} on {model.device} (free was {free_gb:.1f} GB)")
        return model
    except (RuntimeError, torch.cuda.OutOfMemoryError) as e:
        import gc; gc.collect(); torch.cuda.empty_cache()
        model = SentenceTransformer(model_name, device="cpu", **kwargs)
        print(f"  GPU OOM ({type(e).__name__}), loaded {model_name} on CPU")
        return model




def with_gpu_oom_cpu_fallback(model, fn, *args, **kwargs):
    """Run fn(*args, **kwargs). On CUDA OutOfMemoryError, move model to CPU
    and retry once. Use this around encode-heavy calls when a model is loaded
    on GPU but a particular batch (long seq, big batch) might exceed VRAM.
    """
    try:
        return fn(*args, **kwargs)
    except torch.cuda.OutOfMemoryError:
        import gc
        gc.collect(); torch.cuda.empty_cache()
        print(f"  [OOM on GPU] -> moving model to CPU and retrying...")
        try:
            model.to("cpu")
        except Exception as e:
            print(f"  (model.to('cpu') failed: {e})")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return fn(*args, **kwargs)


def load_llm(model_name):
    """Load a HuggingFace causal LLM with its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map='auto',
    )
    gen = hf_pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=False,
    )
    return gen, tokenizer



def strip_base_score(cvss_nl):
    """Remove Base Score from CVSS vector NL string."""
    if not cvss_nl:
        return ""
    return _re.sub(r"\s*Base Score:\s*[\d.]+\.?\s*", "", cvss_nl).strip()

def build_nl_ti_selected(entry_ti):
    """Build TI-selected NL: desc + capec + cvss_vector_nl (no base score)."""
    parts = [entry_ti.get("description", "")]
    capec = entry_ti.get("capec", "")
    if capec:
        parts.append(capec)
    cvss = strip_base_score(entry_ti.get("cvss_vector_nl", ""))
    if cvss:
        parts.append(cvss)
    return " ".join(parts)

cve_descriptions = load_target_pool(TARGET_POOL_FILE)
dataset = load_dataset(DATASET_PATH, cve_descriptions)
bad_dataset = load_bad_dataset(BAD_PDDL_DIR, cve_descriptions)
print(f"Bad (mutated) examples: {len(bad_dataset)} CVEs, {sum(len(e['attack_paths']) for e in bad_dataset)} domains")
prompt_env = load_prompts(PROMPTS_PATH)


embedding_model = None
if eval_flags.embedding_intrinsic or eval_flags.embedding_extrinsic:
    embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)

llm, tokenizer = None, None
if eval_flags.llm_intrinsic or eval_flags.llm_extrinsic:
    llm, tokenizer = load_llm(LLM_MODEL_NAME)

# --- Calibration data for LLM-as-expert evaluation ---
CALIBRATION_DATA_PATH = os.path.join(PROMPTS_PATH, "data.jsonl")

def load_calibration_data(data_jsonl_path, eval_type="intrinsic", exclude_cve=None, n=4, seed=42):
    """Sample n calibration examples from data.jsonl.
    
    Constraints:
        - exclude_cve: the CVE being evaluated is excluded (prevent data leakage)
        - n >= 2: at least 1 good + 1 bad example guaranteed
        - balanced: samples from both calibration_good and calibration_bad pools
    
    Args:
        data_jsonl_path: path to data.jsonl
        eval_type: 'intrinsic' or 'extrinsic'
        exclude_cve: CVE ID to exclude
        n: total number of calibration examples (>= 2)
        seed: random seed for reproducibility
    """
    rng = _rng.Random(seed)
    
    with open(data_jsonl_path) as f:
        all_data = [json.loads(line) for line in f]
    
    pool = [d for d in all_data
            if d.get("eval_type") == eval_type
            and d.get("role", "").startswith("calibration")
            and d.get("cve_id") != exclude_cve]
    
    good = [d for d in pool if d.get("role") == "calibration_good"]
    bad = [d for d in pool if d.get("role") == "calibration_bad"]
    
    n = max(n, 2)  # enforce minimum 2
    n_good = max(1, n // 2)       # at least 1 good
    n_bad = max(1, n - n_good)    # at least 1 bad
    # Adjust if one pool is too small
    n_good = min(n_good, len(good))
    n_bad = min(n_bad, len(bad))
    
    selected_good = rng.sample(good, n_good) if good else []
    selected_bad = rng.sample(bad, n_bad) if bad else []
    
    return selected_good + selected_bad


# --- Rate limit retry wrapper ---
def api_call_with_retry(func, *args, max_retries=5, base_delay=5, **kwargs):
    """Call func with exponential backoff on rate limit errors."""
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                delay = base_delay * (2 ** attempt)
                print(f"  [RATE LIMIT] retry {attempt+1}/{max_retries} in {delay}s...")
                time.sleep(delay)
            else:
                raise
    raise RuntimeError(f"Max retries ({max_retries}) exceeded")


# --- Classification report helpers (following Marco's pattern) ---

def report_row(y_true, y_pred, **meta):
    """Flatten classification_report into a single dict row, with TPR/FPR."""
    rpt = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    row = dict(meta)
    for key, val in rpt.items():
        if isinstance(val, dict):
            for metric, v in val.items():
                row[f'{key}__{metric}'] = v
        else:
            row[key] = val
    row['tpr'] = rpt.get('1', {}).get('recall', float('nan'))
    row['fpr'] = 1.0 - rpt.get('0', {}).get('recall', float('nan'))
    return row

def print_report(y_true, y_pred, title=''):
    if title:
        print(f'\n{title}')
        print('─' * len(title))
    print(classification_report(y_true, y_pred, zero_division=0))


def save_cv_record(cv_path, model_name, threshold, fold_thresholds, row, labels, build_seconds, cv_seconds, cv_method="reference_full_matrix"):
    """Build CV record, dedup by model, and save to JSONL."""
    record = {
        "timestamp": datetime.now().isoformat(),
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "fold_thresholds": fold_thresholds,
        "accuracy": row.get("accuracy", float("nan")),
        "precision": row.get("1__precision", float("nan")),
        "recall": row.get("1__recall", float("nan")),
        "f1": row.get("1__f1-score", float("nan")),
        "tpr": row["tpr"],
        "fpr": row["fpr"],
        "n_positive_pairs": int(labels.sum()),
        "n_negative_pairs": int((labels == 0).sum()),
        "build_pairs_seconds": build_seconds,
        "cv_calibration_seconds": cv_seconds,
    }
    existing = []
    if os.path.exists(cv_path):
        with open(cv_path) as f:
            existing = [json.loads(line) for line in f if line.strip()]
        existing = [r for r in existing if not (r.get("model") == model_name and r.get("cv_method", "reference_full_matrix") == cv_method)]
    existing.append(record)
    with open(cv_path, "w") as f:
        for r in existing:
            f.write(json.dumps(r) + "\n")
    return record


def save_intrinsic_similarity_result(results_path, source, model_name, threshold, cve_id, ap_id, similarity, prediction, elapsed, cv_method="reference_full_matrix"):
    result = {
        "timestamp": datetime.now().isoformat(),
        "source": source,
        "cve_id": cve_id,
        "ap_id": ap_id,
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "similarity": round(similarity, 6),
        "prediction": prediction,
        "elapsed_seconds": round(elapsed, 4),
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_similarity_result(results_path, source, model_name, threshold, cve_id, test_ap, best_ref_ap, similarity, prediction, n_references, elapsed, cv_method="reference_full_matrix"):
    result = {
        "timestamp": datetime.now().isoformat(),
        "source": source,
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "cve_id": cve_id,
        "test_ap": test_ap,
        "best_ref_ap": best_ref_ap,
        "similarity": round(similarity, 6),
        "prediction": prediction,
        "n_references": n_references,
        "elapsed_seconds": round(elapsed, 4),
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_intrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, label, response, usage, price_in, price_out):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "label": label, "response_length": len(response),
        "parse_success": label is not None,
        "llm_response": response,
        "usage": {**usage, "cost_usd": round((usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1000, 6)},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_intrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, verdict, final_score, scores, response, usage, price_in, price_out):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "verdict": verdict, "final_score": final_score, "scores": scores,
        "response_length": len(response),
        "parse_success": "parse_error" not in scores,
        "llm_response": response,
        "usage": {**usage, "cost_usd": round((usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1000, 6)},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, label, n_references, parse_success, usage, elapsed, cost):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "generated": ap_id,
        "label": label, "n_references": n_references,
        "parse_success": parse_success,
        "usage": {**usage, "elapsed_seconds": elapsed, "cost_usd": cost},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, verdict, final_score, n_references, parse_success, usage, elapsed, cost):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "generated": ap_id,
        "verdict": verdict, "final_score": final_score,
        "n_references": n_references,
        "parse_success": parse_success,
        "usage": {**usage, "elapsed_seconds": elapsed, "cost_usd": cost},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def prepare_intrinsic_texts(dataset):
    """Extract texts and build 21x55 pair structure (matches build_intrinsic_pairs).
    Returns:
        descs: 21 unique CVE descriptions (one per CVE)
        domains: 55 reference domains (one per attack path)
        domain_cve_ids: 55 CVE IDs (domain-side, kept for backward compatibility)
        labels: 1155 pair labels (1 if same CVE, else 0), iterated descs-major
        groups: 1155 description-side CVE IDs (for GroupKFold)
    """
    descs = [entry["description"] for entry in dataset]
    desc_cve_ids = [entry["cve_id"] for entry in dataset]
    domains, domain_cve_ids = [], []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            domains.append(ap["domain"])
            domain_cve_ids.append(entry["cve_id"])
    n_d, n_p = len(descs), len(domains)
    labels = np.array([1 if desc_cve_ids[i] == domain_cve_ids[j] else 0
                       for i in range(n_d) for j in range(n_p)])
    groups = np.array([desc_cve_ids[i] for i in range(n_d) for j in range(n_p)])
    return descs, domains, domain_cve_ids, labels, groups


def compute_intrinsic_scores(descs, domains, model):
    """Compute 21x55 similarity scores (flattened, descs-major)."""
    sim_matrix = embedding_similarity_intrinsic(descs, domains, model)
    n_d, n_p = len(descs), len(domains)
    scores = np.array([float(sim_matrix[i, j]) for i in range(n_d) for j in range(n_p)])
    return scores


def prepare_extrinsic_texts(dataset):
    """Extract texts and build pair structure for extrinsic (model-independent)."""
    all_entries = []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            all_entries.append((ap["domain"], entry["cve_id"], ap["ap_id"]))
    domains = [p[0] for p in all_entries]
    cve_ids = [p[1] for p in all_entries]
    n = len(all_entries)
    labels = np.array([1 if cve_ids[i] == cve_ids[j] else 0 for i in range(n) for j in range(i+1, n)])
    groups = np.array([cve_ids[i] for i in range(n) for j in range(i+1, n)])
    return domains, cve_ids, labels, groups


def compute_extrinsic_scores(domains, model):
    """Compute similarity scores for pre-prepared extrinsic texts."""
    E = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    sim_matrix = (E @ E.T).float().cpu().numpy()
    n = len(domains)
    scores = np.array([float(sim_matrix[i, j]) for i in range(n) for j in range(i+1, n)])
    return scores


Bad (mutated) examples: 18 CVEs, 55 domains


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Loaded all-MiniLM-L6-v2 on cuda:0 (free was 3.6 GB)


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Select two example:
1. reference
2. bad one

In [40]:
# ── Build test samples: 1 random reference + 1 random bad ──
BAD_PDDL_DIR = os.path.join(PROJECT_ROOT, "resources", "data", "mutated_bad_PDDL_AP")

rng_test = np.random.default_rng(108)

# Pick 1 random reference AP
ref_entry = dataset[rng_test.integers(len(dataset))]
ref_ap = ref_entry["attack_paths"][rng_test.integers(len(ref_entry["attack_paths"]))]
test_samples = [("reference", ref_entry["cve_id"], ref_ap["ap_id"], ref_ap["domain"])]

# Pick 1 random bad AP
bad_domains = []
for cve_dir in sorted(Path(BAD_PDDL_DIR).iterdir()):
    if not cve_dir.is_dir():
        continue
    for ap_dir in sorted(cve_dir.iterdir()):
        if not ap_dir.is_dir():
            continue
        domain_file = ap_dir / "domain.pddl"
        if domain_file.exists():
            bad_domains.append((cve_dir.name, ap_dir.name, domain_file.read_text(encoding="utf-8")))

bad_pick = bad_domains[rng_test.integers(len(bad_domains))]
test_samples.append(("bad", bad_pick[0], bad_pick[1], bad_pick[2]))

print(f"Test samples: {len(test_samples)}")
for source, cve, ap, dom in test_samples:
    print(f"  [{source}] {cve}/{ap}  ({len(dom)} chars)")

Test samples: 2
  [reference] CVE-2022-1471/AP1  (14884 chars)
  [bad] CVE-2025-22228/AP1_replace_stride_goal_rep1  (12116 chars)


Metric: domain statistics

In [41]:
def domain_stats(domain_pddl):
    """Extract structural stats from a PDDL domain string."""
    return {
        "domain_size_bytes": len(domain_pddl.encode("utf-8")),
        "n_actions": len(re.findall(r"\(:action\s", domain_pddl)),
        "n_predicates": len(re.findall(r"\([\w-]+", re.findall(r"\(:predicates([^)]*(?:\([^)]*\))*[^)]*?)\)", domain_pddl, re.DOTALL)[0])) if re.findall(r"\(:predicates", domain_pddl) else 0,
        "n_types": len(re.findall(r"\(:types([^)]*?)\)", domain_pddl, re.DOTALL)[0].split()) if re.findall(r"\(:types", domain_pddl) else 0,
    }

Metric: peak memory in MB (gpu/cpu)

In [42]:
def get_peak_rss_mb():
    """Get current process peak RSS in MB (cross-platform).
    Unix: uses resource.getrusage (macOS: bytes, Linux: KB).
    Windows / fallback: uses psutil if available, else returns None."""
    if resource is not None:
        ru = resource.getrusage(resource.RUSAGE_CHILDREN)
        if platform.system() == "Darwin":
            return round(ru.ru_maxrss / 1024 / 1024, 2)
        return round(ru.ru_maxrss / 1024, 2)
    try:
        import psutil
        return round(psutil.Process().memory_info().rss / 1024 / 1024, 2)
    except ImportError:
        return None


## 3. Evaluation

### 3.1 Syntax Check (ENHSP)

In [9]:
enhsp = create_enhsp_checker()
# ── Run syntax check ──
t_start_syntax = time.time()
reference_syntax_results = []

save_dir = os.path.join(RESULTS_BASE, "syntax")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_syntax.jsonl")

open(results_path, "w").close()

for source, cve_id, ap_id, domain_pddl in test_samples:
    problem_str = generate_problem(domain_pddl)
    t0 = time.time()
    r_enhsp = enhsp.check_from_string(domain_pddl, problem_str)
    elapsed = time.time() - t0
    stats = domain_stats(domain_pddl)

    result = {"timestamp": datetime.now().isoformat(), 
        "source": source,
        "cve_id": cve_id,
        "ap_id": ap_id,
        "syntax_ok": r_enhsp.success,
        "error": r_enhsp.error,
        "elapsed_seconds": round(elapsed, 4),
        **stats,
    }
    reference_syntax_results.append(result)
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    print(f"[{source}] {cve_id}/{ap_id}  syntax: {r_enhsp.success}  {elapsed:.3f}s")

t_syntax = time.time() - t_start_syntax

n_pass = sum(1 for r in reference_syntax_results if r["syntax_ok"])
print(f"\nSyntax: {n_pass}/{len(reference_syntax_results)} passed, time: {t_syntax:.2f}s")


[reference] CVE-2022-1471/AP1  syntax: False  4.683s
[bad] CVE-2025-22228/AP1_replace_stride_goal_rep1  syntax: False  0.258s

Syntax: 0/2 passed, time: 4.95s


### 3.2 Solvability (Metric-FF)

In [10]:
ff = create_ff_checker()

# ── Run solvability check ──
t_start_solv = time.time()
reference_solvability_results = []

save_dir = os.path.join(RESULTS_BASE, "solvability")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_solvability.jsonl")

open(results_path, "w").close()

for source, cve_id, ap_id, domain_pddl in test_samples:
    problem_str = generate_problem(domain_pddl)
    t0 = time.time()
    r_ff = ff.check_from_string(domain_pddl, problem_str)
    elapsed = time.time() - t0
    stats = domain_stats(domain_pddl)

    result = {"timestamp": datetime.now().isoformat(), 
        "source": source,
        "cve_id": cve_id,
        "ap_id": ap_id,
        "solvable": r_ff.solvable,
        "plan_length": r_ff.plan_length,
        "plan_cost": r_ff.plan_cost,
        "plan": r_ff.plan,
        "error": r_ff.error,
        "elapsed_seconds": round(elapsed, 4),
        **stats,
    }
    reference_solvability_results.append(result)
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    status = "SOLVABLE" if r_ff.solvable else "FAIL"
    print(f"[{source}] {cve_id}/{ap_id}  {status}  plan_length={r_ff.plan_length}  cost={r_ff.plan_cost}  {elapsed:.3f}s")
    if r_ff.plan:
        for i, action in enumerate(r_ff.plan):
            print(f"  {i}: {action}")

t_solv = time.time() - t_start_solv

n_solvable = sum(1 for r in reference_solvability_results if r["solvable"])
print(f"\nSolvability: {n_solvable}/{len(reference_solvability_results)} solvable, time: {t_solv:.2f}s")


[reference] CVE-2022-1471/AP1  SOLVABLE  plan_length=10  cost=12.0  0.163s
  0: ATTACKER-CRAFTS-MALICIOUS-JAVA-CLASS SEFA JAVA-GADGET-CLASS_SEFA COMMAND-EXECUTION-LOGIC_SEFA CVE_2022_1471
  1: ATTACKER-CRAFTS-MALICIOUS-YAML-PAYLOAD SEFA YAML-PAYLOAD_SEFA JAVA-GADGET-CLASS_SEFA JNDI-ENDPOINT_SEFA JAVA-GADGET-CLASS_SEFA CVE_2022_1471
  2: ATTACKER-SENDS-MALICIOUS-YAML-VIA-HTTP-POST HTTP-POST-REQUEST_SEFA SEFA YAML-PAYLOAD_SEFA YAML-ENDPOINT_SEFA CVE_2022_1471
  3: TARGET-SYSTEM-RECEIVES-HTTP-POST-REQUEST-WITH-MALICIOUS-PAYLOAD SEFA HTTP-POST-REQUEST_SEFA YAML-PAYLOAD_SEFA YAML-ENDPOINT_SEFA
  4: TARGET-SYSTEM-PASSES-YAML-TO-SNAKEYAML-LOADER HTTP-POST-REQUEST_SEFA YAML-PAYLOAD_SEFA SEFA SNAKEYAML-LIBRARY_SEFA
  5: TARGET-SYSTEM-STARTS-YAML-DESERIALIZATION-WITH-CONSTRUCTOR YAML-PAYLOAD_SEFA SNAKEYAML-LIBRARY_SEFA YAML-CONSTRUCTOR_SEFA SEFA
  6: TARGET-SYSTEM-INSTANTIATES-DANGEROUS-JAVA-GADGET-CLASS SEFA YAML-CONSTRUCTOR_SEFA JAVA-GADGET-CLASS_SEFA YAML-PAYLOAD_SEFA
  7: TARGET-SYSTEM-LOADS

### 3.3 Semantic Evaluation

#### 3.3.1 Intrinsic

##### 3.3.1.1a Embedding: NL CVE Description vs PDDL Similarity

Block 1:  the function to compute the embedding similarity between the NL CVE description vs the PDDL code (domain)

###### all-MiniLM-L6-v2 

In [43]:
def embedding_similarity_intrinsic(descriptions, pddl_texts, model):
    """Cosine similarity matrix between NL descriptions and PDDL codes.
    Args:
        descriptions: list of CVE NL description strings
        pddl_texts: list of PDDL (domain) strings
        model: SentenceTransformer model
    Returns:
        numpy array of shape (len(descriptions), len(pddl_texts))
    """
    E_desc = model.encode(
        descriptions,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    E_pddl = model.encode(
        pddl_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    S = E_desc @ E_pddl.T
    return S.float().cpu().numpy()

Block 2: Cross-validation, performance estimation, and corrected proportion

Process:
1. Build positive/negative pairs from reference data 
   - 1:PDDL doman vs same CVE NL desc 
   - 0: PDDL doman vs different CVE NL desc
2. CVE-level GroupKFold CV with bootstrap threshold on each train fold
3. Out-of-fold predictions → global confusion matrix → TPR, FPR, classification report

Why GroupKFold?
- one CVE NL desc corresponded to multiple PDDL AP domain 
- If the train fold contains (for example, CVE-2024-12798 NL desc vs domain_AP4) and the validation fold contains (CVE-2024-12798 NL desc vs domain_AP1), the threshold is calibrated on an embedding the validation set also shares, this is data leakage. GroupKFold ensures all pairs involving the same CVE NL desc are assigned to the same fold, eliminating this issue.

In [44]:
# Step 1: Build positive/negative pairs
def build_intrinsic_pairs(dataset, model):
    """
    Build (scores, labels, groups) for intrinsic embedding evaluation.
    Uses 21 unique CVE descriptions × 55 reference domains.
    Positive (1): reference domain × same CVE description
    Negative (0): reference domain × different CVE description
    Groups: description-side CVE ID (for GroupKFold)
    """
    # 21 unique descriptions (one per CVE)
    descs = [entry["description"] for entry in dataset]
    desc_cve_ids = [entry["cve_id"] for entry in dataset]
    
    # 55 reference domains
    domains, domain_cve_ids = [], []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            domains.append(ap["domain"])
            domain_cve_ids.append(entry["cve_id"])
    
    # Encode and compute similarity matrix: 21 × 55
    E_desc = model.encode(descs, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    E_domain = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    sim_matrix = (E_desc @ E_domain.T).float().cpu().numpy()
    
    # Build pairs from matrix
    scores, labels, groups = [], [], []
    for i in range(len(descs)):
        for j in range(len(domains)):
            scores.append(float(sim_matrix[i, j]))
            labels.append(1 if desc_cve_ids[i] == domain_cve_ids[j] else 0)
            groups.append(desc_cve_ids[i])

    return np.array(scores), np.array(labels), np.array(groups)


t_build = time.time()
pair_scores_a, pair_labels_a, pair_groups_a = build_intrinsic_pairs(dataset, embedding_model)
build_pairs_seconds_a = round(time.time() - t_build, 2)
print(f"  Build pairs time: {build_pairs_seconds_a}s")
print(f"Embedding Model: {EMBEDDING_MODEL_NAME}")
print(f"  Positive pairs: {pair_labels_a.sum()}, Negative pairs: {(pair_labels_a == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(pair_groups_a))}")

# Global list for cross-model comparison CSV
calibration_embedding_rows = []


  Build pairs time: 0.14s
Embedding Model: all-MiniLM-L6-v2
  Positive pairs: 55, Negative pairs: 1100
  Groups (CVEs): 21


In [45]:
# Step 2: CV calibration
def _find_threshold_pr(y_true, y_score):
    """Return the threshold closest to (1,1) in the precision-recall curve."""
    prec, rec, thr = precision_recall_curve(y_true, y_score)
    distances = np.sqrt((1 - prec[1:]) ** 2 + (1 - rec[1:]) ** 2)
    return float(thr[np.argmin(distances)])

def run_calibration(scores, labels, groups, k=5, n_bootstrap=500, random_state=42):
    """
    CVE-level GroupKFold CV with bootstrap threshold search on each train fold.
    Returns: median_threshold, fold_thresholds, y_pred (OOF), y_true (OOF)
    """
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    groups = np.asarray(groups)
    rng = np.random.RandomState(random_state)

    unique_groups = np.unique(groups)
    n_groups = len(unique_groups)
    actual_k = min(k, n_groups)
    if actual_k < k:
        print(f"Warning: only {n_groups} groups, reducing k from {k} to {actual_k}")

    gkf = GroupKFold(n_splits=actual_k)
    fold_thresholds, y_pred_parts, y_true_parts = [], [], []

    for fold_i, (train_idx, val_idx) in enumerate(gkf.split(scores, labels, groups)):
        tr_scores, tr_labels = scores[train_idx], labels[train_idx]
        bt = []
        for _ in range(n_bootstrap):
            idx = rng.choice(len(tr_scores), size=len(tr_scores), replace=True)
            bt.append(_find_threshold_pr(tr_labels[idx], tr_scores[idx]))
        fold_thr = float(np.median(bt))
        fold_thresholds.append(fold_thr)
        y_pred_parts.append((scores[val_idx] >= fold_thr).astype(int))
        y_true_parts.append(labels[val_idx])
        val_groups = np.unique(groups[val_idx])
        print(f"  Fold {fold_i+1}: threshold={fold_thr:.4f}, val CVEs={list(val_groups)}")

    return (
        float(np.median(fold_thresholds)),
        fold_thresholds,
        np.concatenate(y_pred_parts),
        np.concatenate(y_true_parts),
    )


t_cv = time.time()
cv_threshold_a, cv_fold_thr_a, cv_pred_a, cv_true_a = run_calibration(
    pair_scores_a, pair_labels_a, pair_groups_a)
cv_calibration_seconds_a = round(time.time() - t_cv, 2)
print(f"CV calibration time: {cv_calibration_seconds_a}s")
print("Fold thresholds:", [f"{t:.4f}" for t in cv_fold_thr_a])
print(f"Median threshold: {cv_threshold_a:.4f}")


  Fold 1: threshold=0.4471, val CVEs=['CVE-2022-1471', 'CVE-2023-33202', 'CVE-2024-22243', 'CVE-2024-38286', 'CVE-2025-24813']
  Fold 2: threshold=0.3730, val CVEs=['CVE-2023-34055', 'CVE-2024-12798', 'CVE-2024-38809', 'CVE-2025-22228']
  Fold 3: threshold=0.4018, val CVEs=['CVE-2022-40149', 'CVE-2023-44487', 'CVE-2024-22259', 'CVE-2024-38816']
  Fold 4: threshold=0.3954, val CVEs=['CVE-2022-40150', 'CVE-2023-46589', 'CVE-2024-22262', 'CVE-2024-38820']
  Fold 5: threshold=0.4457, val CVEs=['CVE-2023-2976', 'CVE-2023-6378', 'CVE-2024-34447', 'CVE-2024-47072']
CV calibration time: 0.86s
Fold thresholds: ['0.4471', '0.3730', '0.4018', '0.3954', '0.4457']
Median threshold: 0.4018


In [46]:
# Step 3: Performance report + save CV metrics

print(f"Embedding Model ({EMBEDDING_MODEL_NAME})")
print(classification_report(cv_true_a, cv_pred_a, zero_division=0))

row_a = report_row(cv_true_a, cv_pred_a,
                   metric="embedding", model=EMBEDDING_MODEL_NAME,
                   mode="intrinsic", threshold=cv_threshold_a)
print(f"TPR = {row_a['tpr']:.4f}  FPR = {row_a['fpr']:.4f}")

# ── Save CV calibration metrics (class 1 only) ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")

cv_record = save_cv_record(cv_path, EMBEDDING_MODEL_NAME, cv_threshold_a, cv_fold_thr_a, row_a, pair_labels_a, build_pairs_seconds_a, cv_calibration_seconds_a)

# Append to global calibration rows
calibration_embedding_rows.append({
    "model": EMBEDDING_MODEL_NAME, "mode": "intrinsic", "threshold": cv_threshold_a,
    "precision": cv_record["precision"], "recall": cv_record["recall"],
    "f1": cv_record["f1"], "tpr": row_a["tpr"], "fpr": row_a["fpr"],
    "n_positive": int(pair_labels_a.sum()), "n_negative": int((pair_labels_a == 0).sum()),
})


Embedding Model (all-MiniLM-L6-v2)
              precision    recall  f1-score   support

           0       0.98      0.89      0.93      1100
           1       0.20      0.55      0.30        55

    accuracy                           0.88      1155
   macro avg       0.59      0.72      0.61      1155
weighted avg       0.94      0.88      0.90      1155

TPR = 0.5455  FPR = 0.1073


In [47]:
# Step 4: Apply threshold to test samples

def apply_threshold_single(description, domain_pddl, model, threshold):
    """Compute intrinsic similarity and apply threshold for a single domain."""
    E_desc = model.encode([description], convert_to_tensor=True, normalize_embeddings=True)
    E_pddl = model.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
    sim = float((E_desc * E_pddl).sum())
    pred = True if sim >= threshold else False
    return sim, pred


# ── Apply to test_samples (1 reference + 1 bad) ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")

open(results_path, "w").close()


print(f"Applying threshold {cv_threshold_a:.4f} ({EMBEDDING_MODEL_NAME}) to test samples:")
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    t0 = time.time()
    sim, pred = apply_threshold_single(description, domain_pddl, embedding_model, cv_threshold_a)
    elapsed = time.time() - t0
    save_intrinsic_similarity_result(results_path, source, EMBEDDING_MODEL_NAME, cv_threshold_a, cve_id, ap_id, sim, pred, elapsed, cv_method="reference_full_matrix")
    print(f"  [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")



# # ── Apply to ALL reference domains (uncomment for full run) ──
# def apply_threshold_batch(descriptions, domains, labels_list, model, threshold):
#     """Apply threshold using batch embedding_similarity_intrinsic. Returns results and PPV."""
#     sim_matrix = embedding_similarity_intrinsic(descriptions, domains, model)
#     results = []
#     for i, label in enumerate(labels_list):
#         sim = float(sim_matrix[i, i])
#         pred = True if sim >= threshold else False
#         results.append({"label": label, "similarity": sim, "prediction": pred})
#         print(f"  {label:45s} prediction: {pred} (similarity: {sim:.4f})")
#     predictions = [r["prediction"] for r in results]
#     ppv = sum(predictions) / len(predictions) if predictions else 0
#     return results, ppv
#
# ref_descs, ref_domains, ref_labels = [], [], []
# for entry in dataset:
#     for ap in entry["attack_paths"]:
#         ref_descs.append(entry["description"])
#         ref_domains.append(ap["domain"])
#         ref_labels.append(f"{entry['cve_id']}/{ap['ap_id']}")
#
# print(f"Applying threshold {cv_threshold_a:.4f} ({EMBEDDING_MODEL_NAME}) to reference domains:")
# ref_results_a, ref_ppv_a = apply_threshold_batch(ref_descs, ref_domains, ref_labels, embedding_model, cv_threshold_a)
# print(f"  Reference PPV: {ref_ppv_a:.4f} ({sum(r['prediction'] for r in ref_results_a)}/{len(ref_results_a)} predicted positive)")
#
# with open(results_path, "w") as f:
#     for r in ref_results_a:
#         r["model"] = EMBEDDING_MODEL_NAME
#         r["threshold"] = cv_threshold_a
#         f.write(json.dumps(r) + "\n")


Applying threshold 0.4018 (all-MiniLM-L6-v2) to test samples:
  [reference] CVE-2022-1471/AP1  sim=0.3815  pred=False  0.026s
  [bad] CVE-2025-22228/AP1_replace_stride_goal_rep1  sim=0.1983  pred=False  0.017s


###### BAAI/bge-base-en-v1.5

In [63]:

EMBEDDING_MODEL_NAME_2 = "BAAI/bge-base-en-v1.5"
embedding_model_2 = load_embedding_model(EMBEDDING_MODEL_NAME_2)
print(f"Loaded second embedding model: {EMBEDDING_MODEL_NAME_2}")

# Step 1: Build pairs
# ── 3.3.1.1a with second embedding model ──build = time.time()
pair_scores_b, pair_labels_b, pair_groups_b = build_intrinsic_pairs(dataset, embedding_model_2)
build_pairs_seconds_b = round(time.time() - t_build, 2)
print(f"  Build pairs time: {build_pairs_seconds_b}s")
print(f"Embedding Model: {EMBEDDING_MODEL_NAME_2}")
print(f"  Positive pairs: {pair_labels_b.sum()}, Negative pairs: {(pair_labels_b == 0).sum()}")

# Step 2: CV calibration
t_cv = time.time()
cv_threshold_b, cv_fold_thr_b, cv_pred_b, cv_true_b = run_calibration(
    pair_scores_b, pair_labels_b, pair_groups_b)
cv_calibration_seconds_b = round(time.time() - t_cv, 2)
print(f"CV calibration time: {cv_calibration_seconds_b}s")
print(f"Median threshold: {cv_threshold_b:.4f}")

# Step 3: Performance report + save CV metrics
print(f"\nEmbedding Model ({EMBEDDING_MODEL_NAME_2})")
print(classification_report(cv_true_b, cv_pred_b, zero_division=0))

row_b = report_row(cv_true_b, cv_pred_b,
                   metric="embedding", model=EMBEDDING_MODEL_NAME_2,
                   mode="intrinsic", threshold=cv_threshold_b)
print(f"TPR = {row_b['tpr']:.4f}  FPR = {row_b['fpr']:.4f}")

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")

cv_record_b = save_cv_record(cv_path, EMBEDDING_MODEL_NAME_2, cv_threshold_b, cv_fold_thr_b, row_b, pair_labels_b, build_pairs_seconds_b, cv_calibration_seconds_b)

calibration_embedding_rows.append({
    "model": EMBEDDING_MODEL_NAME_2, "mode": "intrinsic", "threshold": cv_threshold_b,
    "precision": cv_record_b["precision"], "recall": cv_record_b["recall"],
    "f1": cv_record_b["f1"], "tpr": row_b["tpr"], "fpr": row_b["fpr"],
    "n_positive": int(pair_labels_b.sum()), "n_negative": int((pair_labels_b == 0).sum()),
})

# Step 4: Apply to test samples
results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")

# Append (don't clear — first model's results already in it)
print(f"\nApplying threshold {cv_threshold_b:.4f} ({EMBEDDING_MODEL_NAME_2}) to test samples:")
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    t0 = time.time()
    sim, pred = apply_threshold_single(description, domain_pddl, embedding_model_2, cv_threshold_b)
    elapsed = time.time() - t0
    pred_str = True if sim >= cv_threshold_b else False
    save_intrinsic_similarity_result(results_path, source, EMBEDDING_MODEL_NAME_2, cv_threshold_b, cve_id, ap_id, sim, pred_str, elapsed)
    print(f"  [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred_str}  {elapsed:.3f}s")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Loaded BAAI/bge-base-en-v1.5 on cuda:0 (free was 3.4 GB)
Loaded second embedding model: BAAI/bge-base-en-v1.5
  Build pairs time: 11.47s
Embedding Model: BAAI/bge-base-en-v1.5
  Positive pairs: 55, Negative pairs: 1100
  Fold 1: threshold=0.7264, val CVEs=['CVE-2022-1471', 'CVE-2023-33202', 'CVE-2024-22243', 'CVE-2024-38286', 'CVE-2025-24813']
  Fold 2: threshold=0.7097, val CVEs=['CVE-2023-34055', 'CVE-2024-12798', 'CVE-2024-38809', 'CVE-2025-22228']
  Fold 3: threshold=0.7264, val CVEs=['CVE-2022-40149', 'CVE-2023-44487', 'CVE-2024-22259', 'CVE-2024-38816']
  Fold 4: threshold=0.7105, val CVEs=['CVE-2022-40150', 'CVE-2023-46589', 'CVE-2024-22262', 'CVE-2024-38820']
  Fold 5: threshold=0.7264, val CVEs=['CVE-2023-2976', 'CVE-2023-6378', 'CVE-2024-34447', 'CVE-2024-47072']
CV calibration time: 0.84s
Median threshold: 0.7264

Embedding Model (BAAI/bge-base-en-v1.5)
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1100
           1

###### Qwen-emb-0.6B/4B/8B

In [ ]:
# ── 3.3.1.1a Prepare intrinsic texts (shared across all embedding models) ──
intr_descs, intr_domains, intr_cve_ids, intr_labels, intr_groups = prepare_intrinsic_texts(dataset)
print(f"Intrinsic pairs prepared: {len(intr_descs)} unique descs, {len(intr_domains)} domains, {len(intr_labels)} pairs")
print(f"  Positive: {intr_labels.sum()}, Negative: {(intr_labels == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(intr_groups))}")


In [20]:
import gc, torch

# A) Drop top-level model variables
for name in list(globals()):
    if name.startswith('emb_model_') or name in ('embedding_model', 'embedding_model_2'):
        del globals()[name]
        print(f'  del {name}')

# B) Drop containers that hold strong refs to model objects
for name in list(globals()):
    if name in ('embedding_models_to_test', 'suspects'):
        del globals()[name]
        print(f'  del container: {name}')

# C) Two gc passes (cyclic refs) + cache release
gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f'GPU free: {free_gb:.2f} GB  (reserved={torch.cuda.memory_reserved()/1e9:.2f} GB, allocated={torch.cuda.memory_allocated()/1e9:.2f} GB)')


GPU free: 3.9 GB


In [ ]:
# ── 3.3.1.1a Embedding Model: Qwen/Qwen3-Embedding-0.6B ──
if "emb_model_qwen3_emb_06b" not in dir():
    emb_model_qwen3_emb_06b = load_embedding_model("Qwen/Qwen3-Embedding-0.6B")
EMB_NAME_qwen3_emb_06b = "Qwen/Qwen3-Embedding-0.6B"

# Step 1: Compute scores (batch_size=8 to avoid OOM)
t_build = time.time()
_E_desc = emb_model_qwen3_emb_06b.encode(intr_descs, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True, batch_size=8)
_E_pddl = emb_model_qwen3_emb_06b.encode(intr_domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True, batch_size=8)
_sim_matrix = (_E_desc @ _E_pddl.T).float().cpu().numpy()
del _E_desc, _E_pddl
n_desc, n_pddl = len(intr_descs), len(intr_domains)
scores_qwen3_emb_06b = np.array([float(_sim_matrix[i, j]) for i in range(n_desc) for j in range(n_pddl)])
del _sim_matrix
build_pairs_seconds_qwen3_emb_06b = round(time.time() - t_build, 2)
print(f"{EMB_NAME_qwen3_emb_06b}: build_pairs {build_pairs_seconds_qwen3_emb_06b}s")

# Step 2: CV calibration
t_cv = time.time()
threshold_qwen3_emb_06b, fold_thr_qwen3_emb_06b, cv_pred_qwen3_emb_06b, cv_true_qwen3_emb_06b = run_calibration(
    scores_qwen3_emb_06b, intr_labels, intr_groups)
cv_seconds_qwen3_emb_06b = round(time.time() - t_cv, 2)
print(f"  CV calibration {cv_seconds_qwen3_emb_06b}s, threshold={threshold_qwen3_emb_06b:.4f}")

# Step 3: Performance report + save CV
row_qwen3_emb_06b = report_row(cv_true_qwen3_emb_06b, cv_pred_qwen3_emb_06b,
    metric="embedding", model=EMB_NAME_qwen3_emb_06b, mode="intrinsic", threshold=threshold_qwen3_emb_06b)
print(f"  TPR={row_qwen3_emb_06b['tpr']:.4f}, FPR={row_qwen3_emb_06b['fpr']:.4f}, F1={row_qwen3_emb_06b.get('1__f1-score', 0):.4f}")
print(classification_report(cv_true_qwen3_emb_06b, cv_pred_qwen3_emb_06b, zero_division=0))

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
cv_rec_qwen3_emb_06b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_06b, threshold_qwen3_emb_06b, fold_thr_qwen3_emb_06b,
    row_qwen3_emb_06b, intr_labels, build_pairs_seconds_qwen3_emb_06b, cv_seconds_qwen3_emb_06b)

calibration_embedding_rows.append({
    "model": EMB_NAME_qwen3_emb_06b, "mode": "intrinsic", "threshold": threshold_qwen3_emb_06b,
    "precision": cv_rec_qwen3_emb_06b["precision"], "recall": cv_rec_qwen3_emb_06b["recall"],
    "f1": cv_rec_qwen3_emb_06b["f1"], "tpr": row_qwen3_emb_06b["tpr"], "fpr": row_qwen3_emb_06b["fpr"],
    "n_positive": int(intr_labels.sum()), "n_negative": int((intr_labels == 0).sum()),
})

# Step 4: Test on test_samples
results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")
print(f"\nApplying threshold {threshold_qwen3_emb_06b:.4f} ({EMB_NAME_qwen3_emb_06b}) to test samples:")
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    t0 = time.time()
    E_desc = emb_model_qwen3_emb_06b.encode([description], convert_to_tensor=True, normalize_embeddings=True)
    E_domain = emb_model_qwen3_emb_06b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
    sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
    elapsed = time.time() - t0
    pred = True if sim >= threshold_qwen3_emb_06b else False
    save_intrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_06b, threshold_qwen3_emb_06b, cve_id, ap_id, sim, pred, elapsed, cv_method="reference_full_matrix")
    print(f"  [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

# Free model + reclaim GPU/CPU memory for next model
del emb_model_qwen3_emb_06b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

c:\Users\lincu\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lincu\.cache\huggingface\hub\models--Qwen--Qwen3-Embedding-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

  Loaded Qwen/Qwen3-Embedding-0.6B on cuda:0


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
import gc, torch

# A) Drop top-level model variables
for name in list(globals()):
    if name.startswith('emb_model_') or name in ('embedding_model', 'embedding_model_2'):
        del globals()[name]
        print(f'  del {name}')

# B) Drop containers that hold strong refs to model objects
for name in list(globals()):
    if name in ('embedding_models_to_test', 'suspects'):
        del globals()[name]
        print(f'  del container: {name}')

# C) Two gc passes (cyclic refs) + cache release
gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f'GPU free: {free_gb:.2f} GB  (reserved={torch.cuda.memory_reserved()/1e9:.2f} GB, allocated={torch.cuda.memory_allocated()/1e9:.2f} GB)')


In [ ]:
# ── 3.3.1.1a Embedding Model: Qwen/Qwen3-Embedding-4B ──
if "emb_model_qwen3_emb_4b" not in dir():
    emb_model_qwen3_emb_4b = load_embedding_model("Qwen/Qwen3-Embedding-4B")
EMB_NAME_qwen3_emb_4b = "Qwen/Qwen3-Embedding-4B"

# Step 1: Compute scores (batch_size=8 to avoid OOM)
t_build = time.time()
_E_desc = emb_model_qwen3_emb_4b.encode(intr_descs, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True, batch_size=8)
_E_pddl = emb_model_qwen3_emb_4b.encode(intr_domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True, batch_size=8)
_sim_matrix = (_E_desc @ _E_pddl.T).float().cpu().numpy()
del _E_desc, _E_pddl
n_desc, n_pddl = len(intr_descs), len(intr_domains)
scores_qwen3_emb_4b = np.array([float(_sim_matrix[i, j]) for i in range(n_desc) for j in range(n_pddl)])
del _sim_matrix
build_pairs_seconds_qwen3_emb_4b = round(time.time() - t_build, 2)
print(f"{EMB_NAME_qwen3_emb_4b}: build_pairs {build_pairs_seconds_qwen3_emb_4b}s")

# Step 2: CV calibration
t_cv = time.time()
threshold_qwen3_emb_4b, fold_thr_qwen3_emb_4b, cv_pred_qwen3_emb_4b, cv_true_qwen3_emb_4b = run_calibration(
    scores_qwen3_emb_4b, intr_labels, intr_groups)
cv_seconds_qwen3_emb_4b = round(time.time() - t_cv, 2)
print(f"  CV calibration {cv_seconds_qwen3_emb_4b}s, threshold={threshold_qwen3_emb_4b:.4f}")

# Step 3: Performance report + save CV
row_qwen3_emb_4b = report_row(cv_true_qwen3_emb_4b, cv_pred_qwen3_emb_4b,
    metric="embedding", model=EMB_NAME_qwen3_emb_4b, mode="intrinsic", threshold=threshold_qwen3_emb_4b)
print(f"  TPR={row_qwen3_emb_4b['tpr']:.4f}, FPR={row_qwen3_emb_4b['fpr']:.4f}, F1={row_qwen3_emb_4b.get('1__f1-score', 0):.4f}")
print(classification_report(cv_true_qwen3_emb_4b, cv_pred_qwen3_emb_4b, zero_division=0))

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
cv_rec_qwen3_emb_4b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_4b, threshold_qwen3_emb_4b, fold_thr_qwen3_emb_4b,
    row_qwen3_emb_4b, intr_labels, build_pairs_seconds_qwen3_emb_4b, cv_seconds_qwen3_emb_4b)

calibration_embedding_rows.append({
    "model": EMB_NAME_qwen3_emb_4b, "mode": "intrinsic", "threshold": threshold_qwen3_emb_4b,
    "precision": cv_rec_qwen3_emb_4b["precision"], "recall": cv_rec_qwen3_emb_4b["recall"],
    "f1": cv_rec_qwen3_emb_4b["f1"], "tpr": row_qwen3_emb_4b["tpr"], "fpr": row_qwen3_emb_4b["fpr"],
    "n_positive": int(intr_labels.sum()), "n_negative": int((intr_labels == 0).sum()),
})

# Step 4: Test on test_samples
results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")
print(f"\nApplying threshold {threshold_qwen3_emb_4b:.4f} ({EMB_NAME_qwen3_emb_4b}) to test samples:")
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    t0 = time.time()
    E_desc = emb_model_qwen3_emb_4b.encode([description], convert_to_tensor=True, normalize_embeddings=True)
    E_domain = emb_model_qwen3_emb_4b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
    sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
    elapsed = time.time() - t0
    pred = True if sim >= threshold_qwen3_emb_4b else False
    save_intrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_4b, threshold_qwen3_emb_4b, cve_id, ap_id, sim, pred, elapsed, cv_method="reference_full_matrix")
    print(f"  [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

# Free model + reclaim GPU/CPU memory for next model
del emb_model_qwen3_emb_4b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')


In [ ]:
import gc, torch

# A) Drop top-level model variables
for name in list(globals()):
    if name.startswith('emb_model_') or name in ('embedding_model', 'embedding_model_2'):
        del globals()[name]
        print(f'  del {name}')

# B) Drop containers that hold strong refs to model objects
for name in list(globals()):
    if name in ('embedding_models_to_test', 'suspects'):
        del globals()[name]
        print(f'  del container: {name}')

# C) Two gc passes (cyclic refs) + cache release
gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f'GPU free: {free_gb:.2f} GB  (reserved={torch.cuda.memory_reserved()/1e9:.2f} GB, allocated={torch.cuda.memory_allocated()/1e9:.2f} GB)')


In [ ]:
# Enable HF online so Qwen3-Embedding-8B can download (~16 GB) -- not cached locally
import os
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)
print("HF_HUB_OFFLINE:", os.environ.get("HF_HUB_OFFLINE", "(unset)"))
print("TRANSFORMERS_OFFLINE:", os.environ.get("TRANSFORMERS_OFFLINE", "(unset)"))


In [ ]:
# === 3.3.1.1a Embedding Model: Qwen/Qwen3-Embedding-8B ===
# Default settings (no max_seq truncation, batch_size=8). Requires GPU >= 24GB
# (e.g. A100/H100/4090); will OOM on smaller cards. Run on server.
if "emb_model_qwen3_emb_8b" not in dir():
    emb_model_qwen3_emb_8b = load_embedding_model("Qwen/Qwen3-Embedding-8B")
EMB_NAME_qwen3_emb_8b = "Qwen/Qwen3-Embedding-8B"

# Step 1: Compute scores (batch_size=8 to avoid OOM)
t_build = time.time()
_E_desc = emb_model_qwen3_emb_8b.encode(intr_descs, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True, batch_size=8)
_E_pddl = emb_model_qwen3_emb_8b.encode(intr_domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True, batch_size=8)
_sim_matrix = (_E_desc @ _E_pddl.T).float().cpu().numpy()
del _E_desc, _E_pddl
n_desc, n_pddl = len(intr_descs), len(intr_domains)
scores_qwen3_emb_8b = np.array([float(_sim_matrix[i, j]) for i in range(n_desc) for j in range(n_pddl)])
del _sim_matrix
build_pairs_seconds_qwen3_emb_8b = round(time.time() - t_build, 2)
print(f"{EMB_NAME_qwen3_emb_8b}: build_pairs {build_pairs_seconds_qwen3_emb_8b}s")

# Step 2: CV calibration
t_cv = time.time()
threshold_qwen3_emb_8b, fold_thr_qwen3_emb_8b, cv_pred_qwen3_emb_8b, cv_true_qwen3_emb_8b = run_calibration(
    scores_qwen3_emb_8b, intr_labels, intr_groups)
cv_seconds_qwen3_emb_8b = round(time.time() - t_cv, 2)
print(f"  CV calibration {cv_seconds_qwen3_emb_8b}s, threshold={threshold_qwen3_emb_8b:.4f}")

# Step 3: Performance report + save CV
row_qwen3_emb_8b = report_row(cv_true_qwen3_emb_8b, cv_pred_qwen3_emb_8b,
    metric="embedding", model=EMB_NAME_qwen3_emb_8b, mode="intrinsic", threshold=threshold_qwen3_emb_8b)
print(f"  TPR={row_qwen3_emb_8b['tpr']:.4f}, FPR={row_qwen3_emb_8b['fpr']:.4f}, F1={row_qwen3_emb_8b.get('1__f1-score', 0):.4f}")
print(classification_report(cv_true_qwen3_emb_8b, cv_pred_qwen3_emb_8b, zero_division=0))

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
cv_rec_qwen3_emb_8b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_8b, threshold_qwen3_emb_8b, fold_thr_qwen3_emb_8b,
    row_qwen3_emb_8b, intr_labels, build_pairs_seconds_qwen3_emb_8b, cv_seconds_qwen3_emb_8b)

calibration_embedding_rows.append({
    "model": EMB_NAME_qwen3_emb_8b, "mode": "intrinsic", "threshold": threshold_qwen3_emb_8b,
    "precision": cv_rec_qwen3_emb_8b["precision"], "recall": cv_rec_qwen3_emb_8b["recall"],
    "f1": cv_rec_qwen3_emb_8b["f1"], "tpr": row_qwen3_emb_8b["tpr"], "fpr": row_qwen3_emb_8b["fpr"],
    "n_positive": int(intr_labels.sum()), "n_negative": int((intr_labels == 0).sum()),
})

# Step 4: Test on test_samples
results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")
print(f"
Applying threshold {threshold_qwen3_emb_8b:.4f} ({EMB_NAME_qwen3_emb_8b}) to test samples:")
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    t0 = time.time()
    E_desc = emb_model_qwen3_emb_8b.encode([description], convert_to_tensor=True, normalize_embeddings=True)
    E_domain = emb_model_qwen3_emb_8b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
    sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
    elapsed = time.time() - t0
    pred = True if sim >= threshold_qwen3_emb_8b else False
    save_intrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_8b, threshold_qwen3_emb_8b, cve_id, ap_id, sim, pred, elapsed, cv_method="reference_full_matrix")
    print(f"  [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

# Free model + reclaim GPU/CPU memory for next model
del emb_model_qwen3_emb_8b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')


##### 3.3.1.1c Optimized CV with balanced easy/hard negatives
- Best config from ratio sweep: easy_basic=5, easy_boundary=10, hard=5, total=20 bad + 55 cross-CVE

###### all-MiniLM-L6-v2 

In [ ]:
def build_intrinsic_pairs_good_bad(dataset, bad_dataset, model, batch_size=4):
    """
    Build (scores, labels, groups) using per-sample pairs with good + bad domains.
    Positive (1): reference domain vs its own CVE description
    Negative (0): bad domain vs its CVE description + reference domain vs one random other CVE description
    Groups: CVE ID (for GroupKFold)
    """
    descs, domains, labels, groups = [], [], [], []
    
    # Positive: reference domain + own CVE description
    for entry in dataset:
        for ap in entry["attack_paths"]:
            descs.append(entry["description"])
            domains.append(ap["domain"])
            labels.append(1)
            groups.append(entry["cve_id"])
    
    # Negative type 1: bad domain + own CVE description
    for entry in bad_dataset:
        descs.append(entry["description"])
        domains.append(entry["domain"])
        labels.append(0)
        groups.append(entry["cve_id"])
    
    # Negative type 2: reference domain + one random other CVE description
    all_entries = [(e["cve_id"], e["description"], ap["domain"]) 
                   for e in dataset for ap in e["attack_paths"]]
    rng = np.random.RandomState(42)
    for cve_id, desc, domain in all_entries:
        others = [e for e in dataset if e["cve_id"] != cve_id]
        if others:
            other = others[rng.randint(len(others))]
            descs.append(other["description"])
            domains.append(domain)
            labels.append(0)
            groups.append(cve_id)
    
    # Encode and compute per-pair similarity
    E_desc = model.encode(descs, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False, batch_size=batch_size)
    E_domain = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False, batch_size=batch_size)
    scores = np.array([float((E_desc[i:i+1] @ E_domain[i:i+1].T).float().cpu().numpy()[0, 0]) for i in range(len(descs))])
    
    return scores, np.array(labels), np.array(groups)

Good: 55 domains
Bad: 55 domains

3.3.1.1b Good-vs-Bad CV Calibration (intrinsic)

--- all-MiniLM-L6-v2 ---
  Pairs: 165 (pos=55, neg=110), build=11.7s
  Fold 1: threshold=0.3256, val CVEs=['CVE-2023-33202', 'CVE-2023-34055', 'CVE-2024-12798', 'CVE-2024-38809']
  Fold 2: threshold=0.3670, val CVEs=['CVE-2022-40149', 'CVE-2023-2976', 'CVE-2023-46589', 'CVE-2024-47072']
  Fold 3: threshold=0.3670, val CVEs=['CVE-2023-44487', 'CVE-2024-22243', 'CVE-2024-38286', 'CVE-2025-22228']
  Fold 4: threshold=0.3524, val CVEs=['CVE-2022-1471', 'CVE-2022-40150', 'CVE-2024-22262', 'CVE-2024-38816', 'CVE-2024-38820']
  Fold 5: threshold=0.3184, val CVEs=['CVE-2023-6378', 'CVE-2024-22259', 'CVE-2024-34447', 'CVE-2025-24813']
  Threshold=0.3524, CV=1.63s
  TPR=0.8364, FPR=0.5727, F1=0.5610
  Accuracy=0.5636
              precision    recall  f1-score   support

           0       0.84      0.43      0.57       110
           1       0.42      0.84      0.56        55

    accuracy                        

In [ ]:
# Build bad_dataset list from bad PDDL directory
bad_dataset_for_cv = []
for cve_dir in sorted(Path(BAD_PDDL_DIR).iterdir()):
    if not cve_dir.is_dir():
        continue
    cve_id = cve_dir.name
    description = cve_descriptions.get(cve_id, "")
    for ap_dir in sorted(cve_dir.iterdir()):
        if not ap_dir.is_dir():
            continue
        domain_path = ap_dir / "domain.pddl"
        if domain_path.exists():
            bad_dataset_for_cv.append({
                "cve_id": cve_id,
                "ap_id": ap_dir.name,
                "description": description,
                "domain": domain_path.read_text(),
            })

print(f"Good: {sum(len(e['attack_paths']) for e in dataset)} domains")
print(f"Bad: {len(bad_dataset_for_cv)} domains")

HARD_MUTS = {"delete_random_precondition", "inject_capability_violation", "swap_action_effects", "remove_stride_goal", "corrupt_exposure_action"}
EASY_BOUNDARY_MUTS = {"merge_consecutive_actions", "replace_with_abstract_action"}  # easy but closer to hard boundary
EASY_BASIC_MUTS = {"delete_critical_action", "replace_exploitation_mechanism", "scramble_action_names", "scramble_predicate_names"}

# Classify bad examples into three pools
bad_by_diff = {"hard": [], "easy_boundary": [], "easy_basic": []}
for entry in bad_dataset_for_cv:
    ap_name = entry["ap_id"]
    for m in HARD_MUTS:
        if m in ap_name: bad_by_diff["hard"].append(entry); break
    else:
        for m in EASY_BOUNDARY_MUTS:
            if m in ap_name: bad_by_diff["easy_boundary"].append(entry); break
        else:
            for m in EASY_BASIC_MUTS:
                if m in ap_name: bad_by_diff["easy_basic"].append(entry); break

print(f"Bad by difficulty: hard={len(bad_by_diff['hard'])}, easy_boundary={len(bad_by_diff['easy_boundary'])}, easy_basic={len(bad_by_diff['easy_basic'])}")

# Sample: 5 easy_basic + 10 easy_boundary + 5 hard = 20
rng_opt = np.random.RandomState(SEED)
bad_optimized = (
    [bad_by_diff['easy_basic'][i] for i in rng_opt.choice(len(bad_by_diff['easy_basic']), 5, replace=False)] +
    [bad_by_diff['easy_boundary'][i] for i in rng_opt.choice(len(bad_by_diff['easy_boundary']), 10, replace=False)] +
    [bad_by_diff['hard'][i] for i in rng_opt.choice(len(bad_by_diff['hard']), 5, replace=False)]
)
print(f"Optimized bad subset: {len(bad_optimized)} (eb=5, ebnd=10, h=5)")


NameError: name 'bad_dataset_for_cv' is not defined

In [ ]:
# === 3.3.1.1c Optimized CV — shared setup (per-model cells follow) ===
save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")
opt_thresholds = {}
print(f"3.3.1.1c bad_optimized: {len(bad_optimized)} (eb=5, ebnd=10, h=5 + cross-CVE)")


In [ ]:
if "embedding_model" not in dir():
print(f"\n--- {EMBEDDING_MODEL_NAME} ---")

t_build = time.time()
scores_opt, labels_opt, groups_opt = with_gpu_oom_cpu_fallback(
    embedding_model, build_intrinsic_pairs_good_bad, dataset, bad_optimized, embedding_model
)
build_sec = round(time.time() - t_build, 2)
print(f"  Pairs: {len(scores_opt)} (pos={labels_opt.sum()}, neg={(labels_opt==0).sum()}), build={build_sec}s")

t_cv = time.time()
thr_opt, fold_thr_opt, pred_opt, true_opt = run_calibration(scores_opt, labels_opt, groups_opt)
cv_sec = round(time.time() - t_cv, 2)
print(f"  Threshold={thr_opt:.4f}, CV={cv_sec}s")

row_opt = report_row(true_opt, pred_opt, metric="embedding_optimized",
                    model=EMBEDDING_MODEL_NAME, mode="intrinsic", threshold=thr_opt)
print(f"  TPR={row_opt['tpr']:.4f}, FPR={row_opt['fpr']:.4f}, F1={row_opt.get('1__f1-score', 0):.4f}")
print(f"  Accuracy={row_opt.get('accuracy', 0):.4f}")
print(classification_report(true_opt, pred_opt, zero_division=0))

save_cv_record(cv_path, EMBEDDING_MODEL_NAME, thr_opt, fold_thr_opt, row_opt, labels_opt,
               build_sec, cv_sec, cv_method="control_hardeasy_bad_ratio")
opt_thresholds[EMBEDDING_MODEL_NAME] = thr_opt

# Apply to test samples (same OOM-safe pattern)
def _encode_test_samples():
    print(f"  Applying threshold {thr_opt:.4f} to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        description = cve_descriptions.get(cve_id, "")
        t0 = time.time()
        E_desc = embedding_model.encode([description], convert_to_tensor=True, normalize_embeddings=True)
        E_domain = embedding_model.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
        elapsed = time.time() - t0
        pred = True if sim >= thr_opt else False
        save_intrinsic_similarity_result(results_path, source, EMBEDDING_MODEL_NAME, thr_opt,
                                         cve_id, ap_id, sim, pred, elapsed,
                                         cv_method="control_hardeasy_bad_ratio")
        print(f"    [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

with_gpu_oom_cpu_fallback(embedding_model, _encode_test_samples)



--- sentence-transformers/all-MiniLM-L6-v2 ---


NameError: name 'build_intrinsic_pairs_good_bad' is not defined

In [ ]:
import gc, torch

# A) Drop top-level model variables
for name in list(globals()):
    if name.startswith('emb_model_') or name in ('embedding_model', 'embedding_model_2'):
        del globals()[name]
        print(f'  del {name}')

# B) Drop containers that hold strong refs to model objects
for name in list(globals()):
    if name in ('embedding_models_to_test', 'suspects'):
        del globals()[name]
        print(f'  del container: {name}')

# C) Two gc passes (cyclic refs) + cache release
gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f'GPU free: {free_gb:.2f} GB  (reserved={torch.cuda.memory_reserved()/1e9:.2f} GB, allocated={torch.cuda.memory_allocated()/1e9:.2f} GB)')


###### BAAI/bge-base-en-v1.5

In [62]:
# === 3.3.1.1c Embedding Model: BAAI/bge-base-en-v1.5 ===
if "embedding_model_2" not in dir():
    embedding_model_2 = load_embedding_model("BAAI/bge-base-en-v1.5")
EMBEDDING_MODEL_NAME_2 = "BAAI/bge-base-en-v1.5"
print(f"\n--- {EMBEDDING_MODEL_NAME_2} ---")

t_build = time.time()
scores_opt, labels_opt, groups_opt = with_gpu_oom_cpu_fallback(
    embedding_model_2, build_intrinsic_pairs_good_bad, dataset, bad_optimized, embedding_model_2
)
build_sec = round(time.time() - t_build, 2)
print(f"  Pairs: {len(scores_opt)} (pos={labels_opt.sum()}, neg={(labels_opt==0).sum()}), build={build_sec}s")

t_cv = time.time()
thr_opt, fold_thr_opt, pred_opt, true_opt = run_calibration(scores_opt, labels_opt, groups_opt)
cv_sec = round(time.time() - t_cv, 2)
print(f"  Threshold={thr_opt:.4f}, CV={cv_sec}s")

row_opt = report_row(true_opt, pred_opt, metric="embedding_optimized",
                    model=EMBEDDING_MODEL_NAME_2, mode="intrinsic", threshold=thr_opt)
print(f"  TPR={row_opt['tpr']:.4f}, FPR={row_opt['fpr']:.4f}, F1={row_opt.get('1__f1-score', 0):.4f}")
print(f"  Accuracy={row_opt.get('accuracy', 0):.4f}")
print(classification_report(true_opt, pred_opt, zero_division=0))

save_cv_record(cv_path, EMBEDDING_MODEL_NAME_2, thr_opt, fold_thr_opt, row_opt, labels_opt,
               build_sec, cv_sec, cv_method="control_hardeasy_bad_ratio")
opt_thresholds[EMBEDDING_MODEL_NAME_2] = thr_opt

# Apply to test samples (same OOM-safe pattern)
def _encode_test_samples():
    print(f"  Applying threshold {thr_opt:.4f} to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        description = cve_descriptions.get(cve_id, "")
        t0 = time.time()
        E_desc = embedding_model_2.encode([description], convert_to_tensor=True, normalize_embeddings=True)
        E_domain = embedding_model_2.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
        elapsed = time.time() - t0
        pred = True if sim >= thr_opt else False
        save_intrinsic_similarity_result(results_path, source, EMBEDDING_MODEL_NAME_2, thr_opt,
                                         cve_id, ap_id, sim, pred, elapsed,
                                         cv_method="control_hardeasy_bad_ratio")
        print(f"    [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

with_gpu_oom_cpu_fallback(embedding_model_2, _encode_test_samples)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Loaded BAAI/bge-base-en-v1.5 on cuda:0 (free was 3.7 GB)

--- BAAI/bge-base-en-v1.5 ---


NameError: name 'build_intrinsic_pairs_good_bad' is not defined

In [ ]:
import gc, torch

# A) Drop top-level model variables
for name in list(globals()):
    if name.startswith('emb_model_') or name in ('embedding_model', 'embedding_model_2'):
        del globals()[name]
        print(f'  del {name}')

# B) Drop containers that hold strong refs to model objects
for name in list(globals()):
    if name in ('embedding_models_to_test', 'suspects'):
        del globals()[name]
        print(f'  del container: {name}')

# C) Two gc passes (cyclic refs) + cache release
gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f'GPU free: {free_gb:.2f} GB  (reserved={torch.cuda.memory_reserved()/1e9:.2f} GB, allocated={torch.cuda.memory_allocated()/1e9:.2f} GB)')


###### Qwen3-Embedding-0.6B/4B/8B

In [ ]:
# === 3.3.1.1c Embedding Model: Qwen/Qwen3-Embedding-0.6B ===
if "emb_model_qwen3_emb_06b" not in dir():
    emb_model_qwen3_emb_06b = load_embedding_model("Qwen/Qwen3-Embedding-0.6B")
EMB_NAME_qwen3_emb_06b = "Qwen/Qwen3-Embedding-0.6B"
print(f"\n--- {EMB_NAME_qwen3_emb_06b} ---")

# Cap max_seq_length to avoid attention OOM on long PDDL inputs (seq_len^2 attention mask)
try:
    emb_model_qwen3_emb_06b.max_seq_length = 2048
except Exception:
    pass

t_build = time.time()
scores_opt, labels_opt, groups_opt = with_gpu_oom_cpu_fallback(
    emb_model_qwen3_emb_06b, build_intrinsic_pairs_good_bad, dataset, bad_optimized, emb_model_qwen3_emb_06b
)
build_sec = round(time.time() - t_build, 2)
print(f"  Pairs: {len(scores_opt)} (pos={labels_opt.sum()}, neg={(labels_opt==0).sum()}), build={build_sec}s")

t_cv = time.time()
thr_opt, fold_thr_opt, pred_opt, true_opt = run_calibration(scores_opt, labels_opt, groups_opt)
cv_sec = round(time.time() - t_cv, 2)
print(f"  Threshold={thr_opt:.4f}, CV={cv_sec}s")

row_opt = report_row(true_opt, pred_opt, metric="embedding_optimized",
                    model=EMB_NAME_qwen3_emb_06b, mode="intrinsic", threshold=thr_opt)
print(f"  TPR={row_opt['tpr']:.4f}, FPR={row_opt['fpr']:.4f}, F1={row_opt.get('1__f1-score', 0):.4f}")
print(f"  Accuracy={row_opt.get('accuracy', 0):.4f}")
print(classification_report(true_opt, pred_opt, zero_division=0))

save_cv_record(cv_path, EMB_NAME_qwen3_emb_06b, thr_opt, fold_thr_opt, row_opt, labels_opt,
               build_sec, cv_sec, cv_method="control_hardeasy_bad_ratio")
opt_thresholds[EMB_NAME_qwen3_emb_06b] = thr_opt

# Apply to test samples (same OOM-safe pattern)
def _encode_test_samples():
    print(f"  Applying threshold {thr_opt:.4f} to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        description = cve_descriptions.get(cve_id, "")
        t0 = time.time()
        E_desc = emb_model_qwen3_emb_06b.encode([description], convert_to_tensor=True, normalize_embeddings=True)
        E_domain = emb_model_qwen3_emb_06b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
        elapsed = time.time() - t0
        pred = True if sim >= thr_opt else False
        save_intrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_06b, thr_opt,
                                         cve_id, ap_id, sim, pred, elapsed,
                                         cv_method="control_hardeasy_bad_ratio")
        print(f"    [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_06b, _encode_test_samples)


In [ ]:
import gc, torch

# A) Drop top-level model variables
for name in list(globals()):
    if name.startswith('emb_model_') or name in ('embedding_model', 'embedding_model_2'):
        del globals()[name]
        print(f'  del {name}')

# B) Drop containers that hold strong refs to model objects
for name in list(globals()):
    if name in ('embedding_models_to_test', 'suspects'):
        del globals()[name]
        print(f'  del container: {name}')

# C) Two gc passes (cyclic refs) + cache release
gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f'GPU free: {free_gb:.2f} GB  (reserved={torch.cuda.memory_reserved()/1e9:.2f} GB, allocated={torch.cuda.memory_allocated()/1e9:.2f} GB)')


In [ ]:
# === 3.3.1.1c Embedding Model: Qwen/Qwen3-Embedding-4B (CPU-safe) ===
if "emb_model_qwen3_emb_4b" not in dir():
    emb_model_qwen3_emb_4b = load_embedding_model("Qwen/Qwen3-Embedding-4B")
EMB_NAME_qwen3_emb_4b = "Qwen/Qwen3-Embedding-4B"
print(f"\n--- {EMB_NAME_qwen3_emb_4b} on {emb_model_qwen3_emb_4b.device} ---")

# Cap max_seq_length: tighter than 2048 because Qwen3-Embedding-4B is bigger
try:
    emb_model_qwen3_emb_4b.max_seq_length = 1536
except Exception:
    pass

# CPU-aware batch_size: small to keep RAM/throughput sane on CPU; encode-time cost dominates
_bs = 2
print(f"  encode config: max_seq={emb_model_qwen3_emb_4b.max_seq_length}  batch_size={_bs}")

t_build = time.time()
scores_opt, labels_opt, groups_opt = with_gpu_oom_cpu_fallback(
    emb_model_qwen3_emb_4b,
    build_intrinsic_pairs_good_bad, dataset, bad_optimized, emb_model_qwen3_emb_4b, batch_size=_bs
)
build_sec = round(time.time() - t_build, 2)
print(f"  Pairs: {len(scores_opt)} (pos={labels_opt.sum()}, neg={(labels_opt==0).sum()}), build={build_sec}s")

t_cv = time.time()
thr_opt, fold_thr_opt, pred_opt, true_opt = run_calibration(scores_opt, labels_opt, groups_opt)
cv_sec = round(time.time() - t_cv, 2)
print(f"  Threshold={thr_opt:.4f}, CV={cv_sec}s")

row_opt = report_row(true_opt, pred_opt, metric="embedding_optimized",
                    model=EMB_NAME_qwen3_emb_4b, mode="intrinsic", threshold=thr_opt)
print(f"  TPR={row_opt['tpr']:.4f}, FPR={row_opt['fpr']:.4f}, F1={row_opt.get('1__f1-score', 0):.4f}")
print(f"  Accuracy={row_opt.get('accuracy', 0):.4f}")
print(classification_report(true_opt, pred_opt, zero_division=0))

save_cv_record(cv_path, EMB_NAME_qwen3_emb_4b, thr_opt, fold_thr_opt, row_opt, labels_opt,
               build_sec, cv_sec, cv_method="control_hardeasy_bad_ratio")
opt_thresholds[EMB_NAME_qwen3_emb_4b] = thr_opt

# Apply to test samples (OOM-safe)
def _encode_test_samples():
    print(f"  Applying threshold {thr_opt:.4f} to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        description = cve_descriptions.get(cve_id, "")
        t0 = time.time()
        E_desc = emb_model_qwen3_emb_4b.encode([description], convert_to_tensor=True, normalize_embeddings=True)
        E_domain = emb_model_qwen3_emb_4b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
        elapsed = time.time() - t0
        pred = True if sim >= thr_opt else False
        save_intrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_4b, thr_opt,
                                         cve_id, ap_id, sim, pred, elapsed,
                                         cv_method="control_hardeasy_bad_ratio")
        print(f"    [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_4b, _encode_test_samples)


In [ ]:
import gc, torch

# A) Drop top-level model variables
for name in list(globals()):
    if name.startswith('emb_model_') or name in ('embedding_model', 'embedding_model_2'):
        del globals()[name]
        print(f'  del {name}')

# B) Drop containers that hold strong refs to model objects
for name in list(globals()):
    if name in ('embedding_models_to_test', 'suspects'):
        del globals()[name]
        print(f'  del container: {name}')

# C) Two gc passes (cyclic refs) + cache release
gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f'GPU free: {free_gb:.2f} GB  (reserved={torch.cuda.memory_reserved()/1e9:.2f} GB, allocated={torch.cuda.memory_allocated()/1e9:.2f} GB)')


In [ ]:
# Enable HF online so Qwen3-Embedding-8B can download (~16 GB) -- not cached locally
import os
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)
print("HF_HUB_OFFLINE:", os.environ.get("HF_HUB_OFFLINE", "(unset)"))
print("TRANSFORMERS_OFFLINE:", os.environ.get("TRANSFORMERS_OFFLINE", "(unset)"))


In [ ]:
# === 3.3.1.1c Embedding Model: Qwen/Qwen3-Embedding-8B (CPU-safe) ===
if "emb_model_qwen3_emb_8b" not in dir():
    emb_model_qwen3_emb_8b = load_embedding_model("Qwen/Qwen3-Embedding-8B")
EMB_NAME_qwen3_emb_8b = "Qwen/Qwen3-Embedding-8B"
print(f"\n--- {EMB_NAME_qwen3_emb_8b} on {emb_model_qwen3_emb_8b.device} ---")

# Cap max_seq_length: tighter than 2048 because Qwen3-Embedding-8B is bigger
try:
    emb_model_qwen3_emb_8b.max_seq_length = 1024
except Exception:
    pass

# CPU-aware batch_size: small to keep RAM/throughput sane on CPU; encode-time cost dominates
_bs = 1
print(f"  encode config: max_seq={emb_model_qwen3_emb_8b.max_seq_length}  batch_size={_bs}")

t_build = time.time()
scores_opt, labels_opt, groups_opt = with_gpu_oom_cpu_fallback(
    emb_model_qwen3_emb_8b,
    build_intrinsic_pairs_good_bad, dataset, bad_optimized, emb_model_qwen3_emb_8b, batch_size=_bs
)
build_sec = round(time.time() - t_build, 2)
print(f"  Pairs: {len(scores_opt)} (pos={labels_opt.sum()}, neg={(labels_opt==0).sum()}), build={build_sec}s")

t_cv = time.time()
thr_opt, fold_thr_opt, pred_opt, true_opt = run_calibration(scores_opt, labels_opt, groups_opt)
cv_sec = round(time.time() - t_cv, 2)
print(f"  Threshold={thr_opt:.4f}, CV={cv_sec}s")

row_opt = report_row(true_opt, pred_opt, metric="embedding_optimized",
                    model=EMB_NAME_qwen3_emb_8b, mode="intrinsic", threshold=thr_opt)
print(f"  TPR={row_opt['tpr']:.4f}, FPR={row_opt['fpr']:.4f}, F1={row_opt.get('1__f1-score', 0):.4f}")
print(f"  Accuracy={row_opt.get('accuracy', 0):.4f}")
print(classification_report(true_opt, pred_opt, zero_division=0))

save_cv_record(cv_path, EMB_NAME_qwen3_emb_8b, thr_opt, fold_thr_opt, row_opt, labels_opt,
               build_sec, cv_sec, cv_method="control_hardeasy_bad_ratio")
opt_thresholds[EMB_NAME_qwen3_emb_8b] = thr_opt

# Apply to test samples (OOM-safe)
def _encode_test_samples():
    print(f"  Applying threshold {thr_opt:.4f} to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        description = cve_descriptions.get(cve_id, "")
        t0 = time.time()
        E_desc = emb_model_qwen3_emb_8b.encode([description], convert_to_tensor=True, normalize_embeddings=True)
        E_domain = emb_model_qwen3_emb_8b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
        elapsed = time.time() - t0
        pred = True if sim >= thr_opt else False
        save_intrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_8b, thr_opt,
                                         cve_id, ap_id, sim, pred, elapsed,
                                         cv_method="control_hardeasy_bad_ratio")
        print(f"    [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_8b, _encode_test_samples)


In [ ]:
import gc, torch

# A) Drop top-level model variables
for name in list(globals()):
    if name.startswith('emb_model_') or name in ('embedding_model', 'embedding_model_2'):
        del globals()[name]
        print(f'  del {name}')

# B) Drop containers that hold strong refs to model objects
for name in list(globals()):
    if name in ('embedding_models_to_test', 'suspects'):
        del globals()[name]
        print(f'  del container: {name}')

# C) Two gc passes (cyclic refs) + cache release
gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f'GPU free: {free_gb:.2f} GB  (reserved={torch.cuda.memory_reserved()/1e9:.2f} GB, allocated={torch.cuda.memory_allocated()/1e9:.2f} GB)')


##### 3.3.1.2a Binary Intrinsic (True/False)
`llm_eval_intrinsic_binary`: binary classification using `completion.md.jinja` (binary=True, extrinsic=False)


In [ ]:
# Unified evaluation template (binary/scored × intrinsic/extrinsic via parameters)
eval_template = prompt_env.get_template("completion.md.jinja")

def llm_eval_intrinsic_binary(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """Binary True/False classification: does the PDDL match the CVE?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=512,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


 Block 2: call the function with the code and the specifications from the data set

In [ ]:
# llm_intrinsic_results = []
# eval_set_path = Path(EVAL_SET_DIR)
# for cve_dir in sorted(eval_set_path.iterdir()):
#     if not cve_dir.is_dir():
#         continue
#     cve_id = cve_dir.name
#     description = cve_descriptions.get(cve_id)
#     if description is None:
#         print(f"SKIP {cve_id}: no description in target_pool.json")
#         continue
#     for config_dir in sorted(cve_dir.iterdir()):
#         if not config_dir.is_dir():
#             continue
#         domain_file = config_dir / DOMAIN_FILE
#         problem_file = config_dir / PROBLEM_FILE
#         if not domain_file.exists():
#             continue
#         domain = domain_file.read_text(encoding='utf-8').strip()
#         problem = problem_file.read_text(encoding='utf-8').strip() if problem_file.exists() else ''
#         response = llm_eval_intrinsic(description, domain, problem, llm)
#         llm_intrinsic_results.append({
#             'cve_id': cve_id,
#             'config': config_dir.name,
#             'llm_response': response,
#         })
#         print(f"{cve_id}/{config_dir.name}  response: {response}")

In [ ]:
def preview_intrinsic_binary(n_calibration=2):
    sample = few_shot_pool[0]
    cal_data = load_calibration_data(CALIBRATION_DATA_PATH, 'intrinsic', exclude_cve=sample.cve_id)[:n_calibration]
    render_args = dict(
        binary=True, extrinsic=False,
        calibration_data=cal_data,
        cve_id=sample.cve_id, description=cve_descriptions.get(sample.cve_id, ''), domain=sample.domain_pddl,
    )

    prompt = eval_template.render(**render_args)
    print(f'--- binary intrinsic | {len(cal_data)} calibration examples | {len(prompt)} chars ---')
    print(prompt[:3000])
    if len(prompt) > 3000:
        print(f'\n... [{{len(prompt) - 3000}} chars truncated]')

interact(preview_intrinsic_binary, n_calibration=IntSlider(min=0, max=5, value=0, description='# Cal'));


interactive(children=(IntSlider(value=0, description='# Cal', max=5), Output()), _dom_classes=('widget-interac…

Because waiting more than 200ms didn’t work, add the free Qwen model to quickly validate the pipeline.

NVIDIA - "meta/llama-3.3-70b-instruct"

In [ ]:
nvidia = OpenAI(
    api_key=os.getenv("NVIDIA_API_KEY"),
    base_url="https://integrate.api.nvidia.com/v1",
    timeout=120.0,
)

t_start_llm_bin = time.time()
total_tokens_bin = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
PRICE_PER_1K_IN, PRICE_PER_1K_OUT = 0.0004, 0.0004  # llama-3.3-70b-instruct $/1K tokens
results_path = os.path.join(save_dir, "results_intrinsic_binary.jsonl")

open(results_path, "w").close()

# ── LLM intrinsic binary on reference data (incremental save) ──


# ── Test on test_samples (1 reference + 1 bad) ──
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    response, usage = llm_eval_intrinsic_binary(cve_id, description, domain_pddl, nvidia, NVIDIA_MODEL, seed=SEED)
    
    try:
        text = response.strip()
        if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
        raw_label = json.loads(text).get("label", "?")
        label = True if str(raw_label).lower() in ("true", "1", "yes") else False
    except Exception:
        label = None
    
    save_intrinsic_binary_result(results_path, NVIDIA_MODEL, SEED, 0.0, source, cve_id, ap_id, label, response, usage, PRICE_PER_1K_IN, PRICE_PER_1K_OUT)
    print(f"[{source}] {cve_id}/{ap_id:30s} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")



# llm_intrinsic_binary_results = []
# for entry in dataset:
#     for ap in entry["attack_paths"]:
#         response, usage = llm_eval_intrinsic_binary(entry["cve_id"], entry["description"], ap["domain"], nvidia, NVIDIA_MODEL, seed=SEED)
#         for k in total_tokens_bin: total_tokens_bin[k] += usage.get(k, 0)
#         
#         try:
#             text = response.strip()
#             if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
#             raw_label = json.loads(text).get("label", "?")
#             label = True if str(raw_label).lower() in ("true", "1", "yes") else False
#         except Exception:
#             label = None
#         
#         result = {"llm_model": NVIDIA_MODEL, "cve_id": entry["cve_id"], "ap_id": ap["ap_id"], "label": label, "llm_response": response, "usage": usage}
#         llm_intrinsic_binary_results.append(result)
#         
#         with open(results_path, "a") as f:
#             f.write(json.dumps(result) + "\n")
#         
#         print(f"{entry['cve_id']}/{ap['ap_id']:30s} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")
# 
# t_elapsed_llm_bin = time.time() - t_start_llm_bin
# 
# 
# print(f"\nDone: {len(llm_intrinsic_binary_results)} domains")
# 


[reference] CVE-2022-40149/AP2                            | True | 8.82s  in=4494 out=10
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 | True | 0.91s  in=4953 out=10


gpt-4.1-mini

In [ ]:
load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=120.0,
)

t_start_llm_bin_gpt = time.time()
total_tokens_bin_gpt = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_intrinsic_binary.jsonl")

# NOTE: append to existing file (nvidia results already in it)

# ── LLM intrinsic binary on reference data (GPT, incremental save) ──


# ── Test on test_samples (1 reference + 1 bad) ──
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    response, usage = llm_eval_intrinsic_binary(cve_id, description, domain_pddl, openai_client, GPT_MODEL, seed=SEED)
    
    try:
        text = response.strip()
        if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
        raw_label = json.loads(text).get("label", "?")
        label = True if str(raw_label).lower() in ("true", "1", "yes") else False
    except Exception:
        label = None
    
    save_intrinsic_binary_result(results_path, GPT_MODEL, SEED, 0.0, source, cve_id, ap_id, label, response, usage, PRICE_PER_1K_IN, PRICE_PER_1K_OUT)
    print(f"[{source}] {cve_id}/{ap_id:30s} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")



# llm_intrinsic_binary_results_gpt = []
# for entry in dataset:
#     for ap in entry["attack_paths"]:
#         response, usage = llm_eval_intrinsic_binary(entry["cve_id"], entry["description"], ap["domain"], openai_client, GPT_MODEL, seed=SEED)
#         for k in total_tokens_bin_gpt: total_tokens_bin_gpt[k] += usage.get(k, 0)
#         
#         try:
#             text = response.strip()
#             if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
#             raw_label = json.loads(text).get("label", "?")
#             label = True if str(raw_label).lower() in ("true", "1", "yes") else False
#         except Exception:
#             label = None
#         
#         result = {"llm_model": GPT_MODEL, "cve_id": entry["cve_id"], "ap_id": ap["ap_id"], "label": label, "llm_response": response, "usage": usage}
#         llm_intrinsic_binary_results_gpt.append(result)
#         
#         with open(results_path, "a") as f:
#             f.write(json.dumps(result) + "\n")
#         
#         print(f"{entry['cve_id']}/{ap['ap_id']:30s} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")
# 
# t_elapsed_llm_bin_gpt = time.time() - t_start_llm_bin_gpt
# 
# print(f"\nDone: {len(llm_intrinsic_binary_results_gpt)} domains, {t_elapsed_llm_bin_gpt:.2f}s, Tokens: {total_tokens_bin_gpt}")
# 


[reference] CVE-2022-40149/AP2                            | True | 1.98s  in=4779 out=9
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 | True | 1.53s  in=5278 out=9


##### 3.3.1.2b Scored Intrinsic (12-criteria, integer 0-5)
`llm_eval_intrinsic_scored`: 12-criteria scored evaluation using `completion.md.jinja` (binary=False, extrinsic=False)


In [ ]:
def llm_eval_intrinsic_scored(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """12-criteria scored evaluation (integer 0-5, violation/non-violation scale)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [ ]:
def preview_intrinsic_scored(n_calibration=2):
    sample = few_shot_pool[0]
    cal_data = load_calibration_data(CALIBRATION_DATA_PATH, 'intrinsic', exclude_cve=sample.cve_id)[:n_calibration]
    render_args = dict(
        binary=False, extrinsic=False,
        calibration_data=cal_data,
        cve_id=sample.cve_id, description=cve_descriptions.get(sample.cve_id, ''), domain=sample.domain_pddl,
    )

    prompt = eval_template.render(**render_args)
    print(f'--- scored intrinsic | {len(cal_data)} calibration examples | {len(prompt)} chars ---')
    print(prompt[:3000])
    if len(prompt) > 3000:
        print(f'\n... [{{len(prompt) - 3000}} chars truncated]')

interact(preview_intrinsic_scored, n_calibration=IntSlider(min=0, max=5, value=0, description='# Cal'));


interactive(children=(IntSlider(value=0, description='# Cal', max=5), Output()), _dom_classes=('widget-interac…

Block 2: evaluation result parser and formatter

In [ ]:
SCORED_CRITERIA = ["F1","F2","A1","A2","C1","C2","C3","V1","V2","V3","N1","N2"]

def parse_scored_response(response_text):
    """Parse LLM scored response JSON, extract scores."""
    try:
        text = response_text.strip()
        if text.startswith("```"):
            text = text.split("\n", 1)[1]
            text = text.rsplit("```", 1)[0]
        data = json.loads(text)
        return data
    except Exception:
        return None

def domain_min_score(scores, criteria=SCORED_CRITERIA):
    """Return min of valid criteria scores (0-5). None if no valid scores."""
    valid = [scores[k] for k in criteria if isinstance(scores.get(k), (int, float))]
    return min(valid) if valid else None

def summarize_intrinsic_scored(result):
    """Extract one-line summary from intrinsic scored result."""
    data = parse_scored_response(result["llm_response"])
    if data is None:
        return f"{result['cve_id']}/{result.get('ap_id', result.get('config', '?'))}  PARSE_ERROR"
    qmin = domain_min_score(data)
    scores_str = " | ".join(str(data.get(k, "?")) for k in SCORED_CRITERIA)
    ap = result.get("ap_id", result.get("config", "?"))
    return f"{result['cve_id']}/{ap:30s} | {scores_str} | min={qmin}"

def summarize_extrinsic_scored(result):
    """Extract one-line summary from extrinsic scored result (+ R1-R4)."""
    data = parse_scored_response(result["llm_response"])
    if data is None:
        return f"{result['cve_id']}/{result.get('generated', '?')} vs {result.get('reference', '?')}  PARSE_ERROR"
    ext_keys = SCORED_CRITERIA + ["R1","R2","R3","R4"]
    qmin = domain_min_score(data)
    scores_str = " | ".join(str(data.get(k, "?")) for k in ext_keys)
    gen = result.get("generated", result.get("ap_id", "?"))
    ref = result.get("reference", "?")
    return f"{result['cve_id']}/{gen:20s} vs {ref} | {scores_str} | min={qmin}"


Block 3: call the function with the code and the specifications from the data set

 NVIDIA_MODEL

In [ ]:
t_start_llm_scored = time.time()
total_tokens_scored = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
PRICE_PER_1K_IN, PRICE_PER_1K_OUT = 0.0004, 0.0004  # llama-3.3-70b-instruct $/1K tokens
results_path = os.path.join(save_dir, "results_intrinsic_scored.jsonl")

open(results_path, "w").close()

# ── Test on test_samples (1 reference + 1 bad) ──
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    response, usage = llm_eval_intrinsic_scored(cve_id, description, domain_pddl, nvidia, NVIDIA_MODEL, seed=SEED)
    
    scores = parse_scored_response(response) or {"parse_error": True}
    qmin = domain_min_score(scores) if "parse_error" not in scores else None
    verdict = True if qmin is not None and qmin >= 3 else False
    
    save_intrinsic_scored_result(results_path, NVIDIA_MODEL, SEED, 0.0, source, cve_id, ap_id, verdict, qmin, scores, response, usage, PRICE_PER_1K_IN, PRICE_PER_1K_OUT)
    print(f"[{source}] {cve_id}/{ap_id:30s} | {verdict}({qmin}) | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")

# ── LLM intrinsic scored on reference data ──

# llm_intrinsic_scored_results = []
# for entry in dataset:
#     for ap in entry["attack_paths"]:
#         response, usage = llm_eval_intrinsic_scored(entry["cve_id"], entry["description"], ap["domain"], nvidia, NVIDIA_MODEL, seed=SEED)
#         for k in total_tokens_scored: total_tokens_scored[k] += usage.get(k, 0)
# 
#         scores = parse_scored_response(response) or {"parse_error": True}
#         qmin = domain_min_score(scores) if "parse_error" not in scores else None
#         verdict = True if qmin is not None and qmin >= 3 else False
# 
#         result = {"llm_model": NVIDIA_MODEL, "cve_id": entry["cve_id"], "ap_id": ap["ap_id"], "scores": scores, "min_score": qmin, "verdict": verdict, "llm_response": response, "usage": usage}
#         llm_intrinsic_scored_results.append(result)
# 
#         with open(results_path, "a") as f:
#             f.write(json.dumps(result) + "\n")
# 
#         print(f"{entry['cve_id']}/{ap['ap_id']:30s} | {verdict}({qmin}) | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")
# 
# t_elapsed_llm_scored = time.time() - t_start_llm_scored
# 
# 
# print(f"\nDone: {len(llm_intrinsic_scored_results)} domains, {t_elapsed_llm_scored:.2f}s, Tokens: {total_tokens_scored}")
# 


[reference] CVE-2022-40149/AP2                            | True(5) | 9.6s  in=5843 out=103
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 | True(5) | 6.03s  in=6302 out=103


gpt-4.1-mini

In [ ]:
load_dotenv(override=True)  # override=True Force overwrite existing environment variables.  
key = os.getenv("OPENAI_API_KEY", "")                                                                                   
print(f"Key loaded: {key[:8]}...{key[-4:]}" if len(key) > 12 else "Key NOT found or too short") 

Key loaded: sk-proj-...z4wA


In [ ]:
# openai_client and GPT_MODEL already loaded in 3.3.1.2a binary intrinsic

t_start_llm_gpt = time.time()
total_tokens_gpt = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_intrinsic_scored.jsonl")

# NOTE: append to existing file (nvidia results already in it)

# ── LLM intrinsic scored on reference data (GPT, incremental save) ──


# ── Test on test_samples (1 reference + 1 bad) ──
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    response, usage = llm_eval_intrinsic_scored(cve_id, description, domain_pddl, openai_client, GPT_MODEL, seed=SEED)
    
    scores = parse_scored_response(response) or {"parse_error": True}
    qmin = domain_min_score(scores) if "parse_error" not in scores else None
    verdict = True if qmin is not None and qmin >= 3 else False
    
    save_intrinsic_scored_result(results_path, GPT_MODEL, SEED, 0.0, source, cve_id, ap_id, verdict, qmin, scores, response, usage, PRICE_PER_1K_IN, PRICE_PER_1K_OUT)
    print(f"[{source}] {cve_id}/{ap_id:30s} | {verdict}({qmin}) | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")



# llm_intrinsic_scored_results_gpt = []
# for entry in dataset:
#     for ap in entry["attack_paths"]:
#         response, usage = llm_eval_intrinsic_scored(entry["cve_id"], entry["description"], ap["domain"], openai_client, GPT_MODEL, seed=SEED)
#         for k in total_tokens_gpt: total_tokens_gpt[k] += usage.get(k, 0)
# 
#         scores = parse_scored_response(response) or {"parse_error": True}
#         qmin = domain_min_score(scores) if "parse_error" not in scores else None
#         verdict = True if qmin is not None and qmin >= 3 else False
# 
#         result = {"llm_model": GPT_MODEL, "cve_id": entry["cve_id"], "ap_id": ap["ap_id"], "scores": scores, "min_score": qmin, "verdict": verdict, "llm_response": response, "usage": usage}
#         llm_intrinsic_scored_results_gpt.append(result)
# 
#         with open(results_path, "a") as f:
#             f.write(json.dumps(result) + "\n")
# 
#         print(f"{entry['cve_id']}/{ap['ap_id']:30s} | {verdict}({qmin}) | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")
# 
# t_elapsed_llm_gpt = time.time() - t_start_llm_gpt
# 
# 
# print(f"\nDone: {len(llm_intrinsic_scored_results_gpt)} domains, {t_elapsed_llm_gpt:.2f}s, Tokens: {total_tokens_gpt}")
# 


[reference] CVE-2022-40149/AP2                            | True(5) | 2.74s  in=6138 out=102
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 | True(5) | 2.72s  in=6637 out=102


#### 3.3.2 Extrinsic

##### 3.3.2.1a Embedding: reference PDDL vs generated PDDL Similarity

Block 1: the function to compute the embedding similarity between two blocks of PDDL code (domain + problem)

###### all-MiniLM-L6-v2

In [50]:
def embedding_similarity_extrinsic(pddl_texts_generated, pddl_texts_reference, model):
    """Cosine similarity matrix between generated and reference PDDL codes.
    Args:
        pddl_texts_generated: list of generated PDDL domain strings
        pddl_texts_reference: list of reference PDDL domain strings
        model: SentenceTransformer model
    Returns:
        numpy array of shape (len(generated), len(reference))
    """
    E_generated = model.encode(
        pddl_texts_generated,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    E_reference = model.encode(
        pddl_texts_reference,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    S = E_generated @ E_reference.T
    return S.float().cpu().numpy()

Block 2: Extrinsic cross-validation, performance estimation, and corrected proportion
Same process as intrinsic Block 5 but for extrinsic (domain vs domain):
1. Build positive/negative pairs: 
  - 1：CVE PDDL AP vs the PDDL APs of the same CVE 
  - 0：CVE PDDL AP vs the PDDL APs of the different CVE  
2. CVE-level GroupKFold CV with bootstrap threshold
3. Out-of-fold predictions → classification report → TPR, FPR
4. Apply threshold to reference domains

In [51]:
# ── Step 1: Build extrinsic positive/negative pairs ──
def build_extrinsic_pairs(dataset, model):
    """
    Build (scores, labels, groups) for extrinsic embedding evaluation.
    Positive (1): same CVE, different APs (domain_i vs domain_j)
    Negative (0): different CVE (domain_i vs domain_j)
    Groups: assigned by CVE of first domain in pair
    """
    # Collect all (domain, cve_id) pairs
    all_entries = []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            all_entries.append((ap["domain"], entry["cve_id"], ap["ap_id"]))

    domains = [p[0] for p in all_entries]
    cve_ids = [p[1] for p in all_entries]

    # Batch encode
    E = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    sim_matrix = (E @ E.T).float().cpu().numpy()

    scores, labels, groups = [], [], []
    n = len(all_entries)
    for i in range(n):
        for j in range(i+1, n):  # upper triangle only, symmetric
            sim = float(sim_matrix[i, j])
            label = 1 if cve_ids[i] == cve_ids[j] else 0
            scores.append(sim)
            labels.append(label)
            groups.append(cve_ids[i])  # group by first domain's CVE

    return np.array(scores), np.array(labels), np.array(groups)


t_build = time.time()
ext_scores_a, ext_labels_a, ext_groups_a = build_extrinsic_pairs(dataset, embedding_model)
ext_build_pairs_seconds_a = round(time.time() - t_build, 2)
print(f"  Build pairs time: {ext_build_pairs_seconds_a}s")
print(f"Extrinsic Embedding Model: {EMBEDDING_MODEL_NAME}")
print(f"  Positive pairs: {ext_labels_a.sum()}, Negative pairs: {(ext_labels_a == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(ext_groups_a))}")


  Build pairs time: 0.11s
Extrinsic Embedding Model: sentence-transformers/all-MiniLM-L6-v2
  Positive pairs: 74, Negative pairs: 1411
  Groups (CVEs): 21


In [52]:
# ── Step 2: Extrinsic CV calibration ──

t_cv = time.time()
ext_cv_threshold_a, ext_cv_fold_thr_a, ext_cv_pred_a, ext_cv_true_a = run_calibration(
    ext_scores_a, ext_labels_a, ext_groups_a)
ext_cv_calibration_seconds_a = round(time.time() - t_cv, 2)
print(f"CV calibration time: {ext_cv_calibration_seconds_a}s")
print(f"\nMedian threshold: {ext_cv_threshold_a:.4f}")
print("Fold thresholds:", [f"{t:.4f}" for t in ext_cv_fold_thr_a])


  Fold 1: threshold=0.9621, val CVEs=['CVE-2022-1471', 'CVE-2024-12798', 'CVE-2024-38809', 'CVE-2025-24813']
  Fold 2: threshold=0.9744, val CVEs=['CVE-2023-44487', 'CVE-2023-46589', 'CVE-2024-22262', 'CVE-2024-38820', 'CVE-2024-47072']
  Fold 3: threshold=0.9383, val CVEs=['CVE-2022-40150', 'CVE-2023-2976', 'CVE-2023-6378', 'CVE-2024-38816']
  Fold 4: threshold=0.9621, val CVEs=['CVE-2024-22243', 'CVE-2024-22259', 'CVE-2024-34447', 'CVE-2024-38286']
  Fold 5: threshold=0.9621, val CVEs=['CVE-2022-40149', 'CVE-2023-33202', 'CVE-2023-34055', 'CVE-2025-22228']
CV calibration time: 0.95s

Median threshold: 0.9621
Fold thresholds: ['0.9621', '0.9744', '0.9383', '0.9621', '0.9621']


In [54]:
# ── Step 3: Extrinsic performance report + save CV metrics ──
print(f"Extrinsic Embedding Model ({EMBEDDING_MODEL_NAME})")
print(classification_report(ext_cv_true_a, ext_cv_pred_a, zero_division=0))

ext_row_a = report_row(ext_cv_true_a, ext_cv_pred_a,
                      metric="embedding", model=EMBEDDING_MODEL_NAME,
                      mode="extrinsic", threshold=ext_cv_threshold_a)
print(f"TPR = {ext_row_a['tpr']:.4f}  FPR = {ext_row_a['fpr']:.4f}")

# ── Save CV calibration metrics (class 1 only) ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")

ext_cv_record = save_cv_record(cv_path, EMBEDDING_MODEL_NAME, ext_cv_threshold_a, ext_cv_fold_thr_a, ext_row_a, ext_labels_a, ext_build_pairs_seconds_a, ext_cv_calibration_seconds_a)


calibration_embedding_rows.append({
    "model": EMBEDDING_MODEL_NAME, "mode": "extrinsic", "threshold": ext_cv_threshold_a,
    "precision": ext_cv_record["precision"], "recall": ext_cv_record["recall"],
    "f1": ext_cv_record["f1"], "tpr": ext_row_a["tpr"], "fpr": ext_row_a["fpr"],
    "n_positive": int(ext_labels_a.sum()), "n_negative": int((ext_labels_a == 0).sum()),
})


Extrinsic Embedding Model (sentence-transformers/all-MiniLM-L6-v2)
              precision    recall  f1-score   support

           0       1.00      0.94      0.97      1411
           1       0.45      0.92      0.61        74

    accuracy                           0.94      1485
   macro avg       0.72      0.93      0.79      1485
weighted avg       0.97      0.94      0.95      1485

TPR = 0.9189  FPR = 0.0581


In [55]:
# ── Step 4: Apply extrinsic threshold to test samples ──

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_extrinsic_similarity.jsonl")

open(results_path, "w").close()

# For extrinsic, compare each test sample against reference APs of the same CVE

print(f"Applying threshold {ext_cv_threshold_a:.4f} ({EMBEDDING_MODEL_NAME}) to test samples:")
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}

for source, cve_id, ap_id, domain_pddl in test_samples:
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"  [{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    t0 = time.time()
    E_test = embedding_model.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
    ref_texts = [ap["domain"] for ap in ref_aps]
    E_ref = embedding_model.encode(ref_texts, convert_to_tensor=True, normalize_embeddings=True)
    sims = (E_test @ E_ref.T).float().cpu().numpy()[0]
    elapsed = time.time() - t0
    # Find best matching reference(s) (max similarity)
    best_sim = float(sims.max())
    best_js = [j for j in range(len(sims)) if float(sims[j]) == best_sim]
    pred = True if best_sim >= ext_cv_threshold_a else False
    for j, ref_ap in enumerate(ref_aps):
        sim = float(sims[j])
        p = True if sim >= ext_cv_threshold_a else False
        print(f"  [{source}] {cve_id}/{ap_id} vs {ref_ap['ap_id']}  sim={sim:.4f}  pred={p}  {elapsed:.3f}s")
    save_extrinsic_similarity_result(results_path, source, EMBEDDING_MODEL_NAME, ext_cv_threshold_a, cve_id, ap_id, [ref_aps[j]["ap_id"] for j in best_js], best_sim, pred, len(ref_aps), elapsed)
    print(f"  [{source}] {cve_id}/{ap_id} BEST={[ref_aps[j]['ap_id'] for j in best_js]}  sim={best_sim:.4f}  pred={pred}")



# # ── Apply to ALL reference domain pairs (uncomment for full run) ──
# ext_ref_results_a = []
# for entry in dataset:
#     ref_aps = entry["attack_paths"]
#     if len(ref_aps) < 2:
#         continue
#     texts = [ap["domain"] for ap in ref_aps]
#     E = embedding_model.encode(texts, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
#     S = (E @ E.T).float().cpu().numpy()
#     for i in range(len(ref_aps)):
#         for j in range(i+1, len(ref_aps)):
#             sim = float(S[i, j])
#             pred = True if sim >= ext_cv_threshold_a else False
#             label = f"{entry['cve_id']}/{ref_aps[i]['ap_id']} vs {ref_aps[j]['ap_id']}"
#             ext_ref_results_a.append({"label": label, "similarity": sim, "prediction": pred})
#             print(f"  {label:45s} prediction: {pred} (similarity: {sim:.4f})")
#
# ext_ref_ppv_a = sum(r["prediction"] for r in ext_ref_results_a) / len(ext_ref_results_a) if ext_ref_results_a else 0
# print(f"  Extrinsic Reference PPV: {ext_ref_ppv_a:.4f}")
#
# with open(results_path, "w") as f:
#     for r in ext_ref_results_a:
#         r["model"] = EMBEDDING_MODEL_NAME
#         r["threshold"] = ext_cv_threshold_a
#         f.write(json.dumps(r) + "\n")


Applying threshold 0.9621 (sentence-transformers/all-MiniLM-L6-v2) to test samples:
  [reference] CVE-2022-1471/AP1 vs AP1  sim=1.0000  pred=True  0.031s
  [reference] CVE-2022-1471/AP1 BEST=['AP1']  sim=1.0000  pred=True
  [bad] CVE-2025-22228/AP1_replace_stride_goal_rep1 vs AP1  sim=1.0000  pred=True  0.032s
  [bad] CVE-2025-22228/AP1_replace_stride_goal_rep1 vs AP2  sim=1.0000  pred=True  0.032s
  [bad] CVE-2025-22228/AP1_replace_stride_goal_rep1 vs AP3  sim=1.0000  pred=True  0.032s
  [bad] CVE-2025-22228/AP1_replace_stride_goal_rep1 BEST=['AP1', 'AP2', 'AP3']  sim=1.0000  pred=True


###### bge-base-en-v1.5

In [67]:
import gc, torch

# A) Drop top-level model variables
for name in list(globals()):
    if name.startswith('emb_model_') or name in ('embedding_model'):
        del globals()[name]
        print(f'  del {name}')

# B) Drop containers that hold strong refs to model objects
for name in list(globals()):
    if name in ('embedding_models_to_test', 'suspects'):
        del globals()[name]
        print(f'  del container: {name}')

# C) Two gc passes (cyclic refs) + cache release
gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f'GPU free: {free_gb:.2f} GB  (reserved={torch.cuda.memory_reserved()/1e9:.2f} GB, allocated={torch.cuda.memory_allocated()/1e9:.2f} GB)')


GPU free: 3.20 GB  (reserved=4.17 GB, allocated=3.87 GB)


In [68]:
# ── 3.3.2.1a with second embedding model ──
# embedding_model_2 already loaded in intrinsic section
# Step 1: Compute scores with model 2 (same pairs as model 1, re-encoded)
t_build = time.time()
ext_scores_b, ext_labels_b, ext_groups_b = build_extrinsic_pairs(dataset, embedding_model_2)
ext_build_pairs_seconds_b = round(time.time() - t_build, 2)
print(f"  Build pairs time: {ext_build_pairs_seconds_b}s")
print(f"Extrinsic Embedding Model: {EMBEDDING_MODEL_NAME_2}")
print(f"  Positive pairs: {ext_labels_b.sum()}, Negative pairs: {(ext_labels_b == 0).sum()}")

# Step 2: CV calibration
t_cv = time.time()
ext_cv_threshold_b, ext_cv_fold_thr_b, ext_cv_pred_b, ext_cv_true_b = run_calibration(
    ext_scores_b, ext_labels_b, ext_groups_b)
ext_cv_calibration_seconds_b = round(time.time() - t_cv, 2)
print(f"CV calibration time: {ext_cv_calibration_seconds_b}s")
print(f"Median threshold: {ext_cv_threshold_b:.4f}")

# Step 3: Performance report + save CV metrics
print(f"\nExtrinsic Embedding Model ({EMBEDDING_MODEL_NAME_2})")
print(classification_report(ext_cv_true_b, ext_cv_pred_b, zero_division=0))

ext_row_b = report_row(ext_cv_true_b, ext_cv_pred_b,
                      metric="embedding", model=EMBEDDING_MODEL_NAME_2,
                      mode="extrinsic", threshold=ext_cv_threshold_b)
print(f"TPR = {ext_row_b['tpr']:.4f}  FPR = {ext_row_b['fpr']:.4f}")

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")

ext_cv_record_b = save_cv_record(cv_path, EMBEDDING_MODEL_NAME_2, ext_cv_threshold_b, ext_cv_fold_thr_b, ext_row_b, ext_labels_b, ext_build_pairs_seconds_b, ext_cv_calibration_seconds_b)

calibration_embedding_rows.append({
    "model": EMBEDDING_MODEL_NAME_2, "mode": "extrinsic", "threshold": ext_cv_threshold_b,
    "precision": ext_cv_record_b["precision"], "recall": ext_cv_record_b["recall"],
    "f1": ext_cv_record_b["f1"], "tpr": ext_row_b["tpr"], "fpr": ext_row_b["fpr"],
    "n_positive": int(ext_labels_b.sum()), "n_negative": int((ext_labels_b == 0).sum()),
})

# Step 4: Apply to test samples
results_path = os.path.join(save_dir, "results_extrinsic_similarity.jsonl")

# Append
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
print(f"\nApplying threshold {ext_cv_threshold_b:.4f} ({EMBEDDING_MODEL_NAME_2}) to test samples:")
for source, cve_id, ap_id, domain_pddl in test_samples:
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"  [{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    t0 = time.time()
    E_test = embedding_model_2.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
    ref_texts = [ap["domain"] for ap in ref_aps]
    E_ref = embedding_model_2.encode(ref_texts, convert_to_tensor=True, normalize_embeddings=True)
    sims = (E_test @ E_ref.T).float().cpu().numpy()[0]
    elapsed = time.time() - t0
    # Find best matching reference(s) (max similarity)
    best_sim = float(sims.max())
    best_js = [j for j in range(len(sims)) if float(sims[j]) == best_sim]
    pred = True if best_sim >= ext_cv_threshold_b else False
    for j, ref_ap in enumerate(ref_aps):
        sim = float(sims[j])
        p = True if sim >= ext_cv_threshold_b else False
        print(f"  [{source}] {cve_id}/{ap_id} vs {ref_ap['ap_id']}  sim={sim:.4f}  pred={p}  {elapsed:.3f}s")
    save_extrinsic_similarity_result(results_path, source, EMBEDDING_MODEL_NAME_2, ext_cv_threshold_b, cve_id, ap_id, [ref_aps[j]["ap_id"] for j in best_js], best_sim, pred, len(ref_aps), elapsed)
    print(f"  [{source}] {cve_id}/{ap_id} BEST={[ref_aps[j]['ap_id'] for j in best_js]}  sim={best_sim:.4f}  pred={pred}")


  Build pairs time: 0.93s
Extrinsic Embedding Model: BAAI/bge-base-en-v1.5
  Positive pairs: 74, Negative pairs: 1411
  Fold 1: threshold=0.9923, val CVEs=['CVE-2022-1471', 'CVE-2024-12798', 'CVE-2024-38809', 'CVE-2025-24813']
  Fold 2: threshold=0.9936, val CVEs=['CVE-2023-44487', 'CVE-2023-46589', 'CVE-2024-22262', 'CVE-2024-38820', 'CVE-2024-47072']
  Fold 3: threshold=0.9927, val CVEs=['CVE-2022-40150', 'CVE-2023-2976', 'CVE-2023-6378', 'CVE-2024-38816']
  Fold 4: threshold=0.9728, val CVEs=['CVE-2024-22243', 'CVE-2024-22259', 'CVE-2024-34447', 'CVE-2024-38286']
  Fold 5: threshold=0.9927, val CVEs=['CVE-2022-40149', 'CVE-2023-33202', 'CVE-2023-34055', 'CVE-2025-22228']
CV calibration time: 0.96s
Median threshold: 0.9927

Extrinsic Embedding Model (BAAI/bge-base-en-v1.5)
              precision    recall  f1-score   support

           0       0.99      0.94      0.97      1411
           1       0.46      0.89      0.60        74

    accuracy                           0.94      1

###### Qwen-emb-0.6B/4B/8B

In [69]:
import gc, torch

# A) Drop top-level model variables
for name in list(globals()):
    if name.startswith('emb_model_') or name in ('embedding_model', 'embedding_model_2'):
        del globals()[name]
        print(f'  del {name}')

# B) Drop containers that hold strong refs to model objects
for name in list(globals()):
    if name in ('embedding_models_to_test', 'suspects'):
        del globals()[name]
        print(f'  del container: {name}')

# C) Two gc passes (cyclic refs) + cache release
gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f'GPU free: {free_gb:.2f} GB  (reserved={torch.cuda.memory_reserved()/1e9:.2f} GB, allocated={torch.cuda.memory_allocated()/1e9:.2f} GB)')


  del embedding_model_2
GPU free: 3.69 GB  (reserved=3.68 GB, allocated=3.43 GB)


In [ ]:
# ── 3.3.2.1a Prepare extrinsic texts (shared across all embedding models) ──
extr_domains, extr_cve_ids, extr_labels, extr_groups = prepare_extrinsic_texts(dataset)
print(f"Extrinsic pairs prepared: {len(extr_domains)} domains, {len(extr_labels)} pairs")
print(f"  Positive: {extr_labels.sum()}, Negative: {(extr_labels == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(extr_groups))}")


Extrinsic pairs prepared: 55 domains, 1485 pairs
  Positive: 74, Negative: 1411
  Groups (CVEs): 21


In [ ]:
# === 3.3.2.1a Embedding Model: Qwen/Qwen3-Embedding-0.6B ===
# Reload if intrinsic section already freed the model
if "emb_model_qwen3_emb_06b" not in dir():
    emb_model_qwen3_emb_06b = load_embedding_model("Qwen/Qwen3-Embedding-0.6B")
EMB_NAME_qwen3_emb_06b = "Qwen/Qwen3-Embedding-0.6B"

# Step 1: Compute scores
t_build = time.time()
_E = emb_model_qwen3_emb_06b.encode(extr_domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True, batch_size=8)
_sim_matrix = (_E @ _E.T).float().cpu().numpy()
del _E
n_ext = len(extr_domains)
ext_scores_qwen3_emb_06b = np.array([float(_sim_matrix[i, j]) for i in range(n_ext) for j in range(i+1, n_ext)])
del _sim_matrix
ext_build_seconds_qwen3_emb_06b = round(time.time() - t_build, 2)
print(f"{EMB_NAME_qwen3_emb_06b}: build_pairs {ext_build_seconds_qwen3_emb_06b}s")

# Step 2: CV calibration
t_cv = time.time()
ext_threshold_qwen3_emb_06b, ext_fold_thr_qwen3_emb_06b, ext_cv_pred_qwen3_emb_06b, ext_cv_true_qwen3_emb_06b = run_calibration(
    ext_scores_qwen3_emb_06b, extr_labels, extr_groups)
ext_cv_seconds_qwen3_emb_06b = round(time.time() - t_cv, 2)
print(f"  CV calibration {ext_cv_seconds_qwen3_emb_06b}s, threshold={ext_threshold_qwen3_emb_06b:.4f}")

# Step 3: Performance report + save CV
ext_row_qwen3_emb_06b = report_row(ext_cv_true_qwen3_emb_06b, ext_cv_pred_qwen3_emb_06b,
    metric="embedding", model=EMB_NAME_qwen3_emb_06b, mode="extrinsic", threshold=ext_threshold_qwen3_emb_06b)
print(f"  TPR={ext_row_qwen3_emb_06b['tpr']:.4f}, FPR={ext_row_qwen3_emb_06b['fpr']:.4f}, F1={ext_row_qwen3_emb_06b.get('1__f1-score', 0):.4f}")
print(classification_report(ext_cv_true_qwen3_emb_06b, ext_cv_pred_qwen3_emb_06b, zero_division=0))

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")
ext_cv_rec_qwen3_emb_06b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_06b, ext_threshold_qwen3_emb_06b, ext_fold_thr_qwen3_emb_06b,
    ext_row_qwen3_emb_06b, extr_labels, ext_build_seconds_qwen3_emb_06b, ext_cv_seconds_qwen3_emb_06b)

calibration_embedding_rows.append({
    "model": EMB_NAME_qwen3_emb_06b, "mode": "extrinsic", "threshold": ext_threshold_qwen3_emb_06b,
    "precision": ext_cv_rec_qwen3_emb_06b["precision"], "recall": ext_cv_rec_qwen3_emb_06b["recall"],
    "f1": ext_cv_rec_qwen3_emb_06b["f1"], "tpr": ext_row_qwen3_emb_06b["tpr"], "fpr": ext_row_qwen3_emb_06b["fpr"],
    "n_positive": int(extr_labels.sum()), "n_negative": int((extr_labels == 0).sum()),
})

# Step 4: Test on test_samples
results_path = os.path.join(save_dir, "results_extrinsic_similarity.jsonl")
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
print(f"Applying threshold {ext_threshold_qwen3_emb_06b:.4f} ({EMB_NAME_qwen3_emb_06b}) to test samples:")
for source, cve_id, ap_id, domain_pddl in test_samples:
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"  [{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    t0 = time.time()
    E_test = emb_model_qwen3_emb_06b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
    ref_texts = [ap["domain"] for ap in ref_aps]
    E_ref = emb_model_qwen3_emb_06b.encode(ref_texts, convert_to_tensor=True, normalize_embeddings=True)
    sims = (E_test @ E_ref.T).float().cpu().numpy()[0]
    elapsed = time.time() - t0
    best_sim = float(sims.max())
    best_js = [j for j in range(len(sims)) if float(sims[j]) == best_sim]
    pred = True if best_sim >= ext_threshold_qwen3_emb_06b else False
    for j, ref_ap in enumerate(ref_aps):
        sim = float(sims[j])
        p = True if sim >= ext_threshold_qwen3_emb_06b else False
        print(f"  [{source}] {cve_id}/{ap_id} vs {ref_ap['ap_id']}  sim={sim:.4f}  pred={p}  {elapsed:.3f}s")
    save_extrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_06b, ext_threshold_qwen3_emb_06b, cve_id, ap_id,
        [ref_aps[j]["ap_id"] for j in best_js], best_sim, pred, len(ref_aps), elapsed)
    print(f"  [{source}] {cve_id}/{ap_id} BEST={[ref_aps[j]['ap_id'] for j in best_js]}  sim={best_sim:.4f}  pred={pred}")

# Free model + reclaim GPU/CPU memory for next model
del emb_model_qwen3_emb_06b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  Loaded Qwen/Qwen3-Embedding-0.6B on cuda:0 (free was 3.7 GB)


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
# === 3.3.2.1a Embedding Model: Qwen/Qwen3-Embedding-4B ===
# Reload if intrinsic section already freed the model
if "emb_model_qwen3_emb_4b" not in dir():
    emb_model_qwen3_emb_4b = load_embedding_model("Qwen/Qwen3-Embedding-4B")
EMB_NAME_qwen3_emb_4b = "Qwen/Qwen3-Embedding-4B"

# Step 1: Compute scores
t_build = time.time()
_E = emb_model_qwen3_emb_4b.encode(extr_domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True, batch_size=8)
_sim_matrix = (_E @ _E.T).float().cpu().numpy()
del _E
n_ext = len(extr_domains)
ext_scores_qwen3_emb_4b = np.array([float(_sim_matrix[i, j]) for i in range(n_ext) for j in range(i+1, n_ext)])
del _sim_matrix
ext_build_seconds_qwen3_emb_4b = round(time.time() - t_build, 2)
print(f"{EMB_NAME_qwen3_emb_4b}: build_pairs {ext_build_seconds_qwen3_emb_4b}s")

# Step 2: CV calibration
t_cv = time.time()
ext_threshold_qwen3_emb_4b, ext_fold_thr_qwen3_emb_4b, ext_cv_pred_qwen3_emb_4b, ext_cv_true_qwen3_emb_4b = run_calibration(
    ext_scores_qwen3_emb_4b, extr_labels, extr_groups)
ext_cv_seconds_qwen3_emb_4b = round(time.time() - t_cv, 2)
print(f"  CV calibration {ext_cv_seconds_qwen3_emb_4b}s, threshold={ext_threshold_qwen3_emb_4b:.4f}")

# Step 3: Performance report + save CV
ext_row_qwen3_emb_4b = report_row(ext_cv_true_qwen3_emb_4b, ext_cv_pred_qwen3_emb_4b,
    metric="embedding", model=EMB_NAME_qwen3_emb_4b, mode="extrinsic", threshold=ext_threshold_qwen3_emb_4b)
print(f"  TPR={ext_row_qwen3_emb_4b['tpr']:.4f}, FPR={ext_row_qwen3_emb_4b['fpr']:.4f}, F1={ext_row_qwen3_emb_4b.get('1__f1-score', 0):.4f}")
print(classification_report(ext_cv_true_qwen3_emb_4b, ext_cv_pred_qwen3_emb_4b, zero_division=0))

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")
ext_cv_rec_qwen3_emb_4b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_4b, ext_threshold_qwen3_emb_4b, ext_fold_thr_qwen3_emb_4b,
    ext_row_qwen3_emb_4b, extr_labels, ext_build_seconds_qwen3_emb_4b, ext_cv_seconds_qwen3_emb_4b)

calibration_embedding_rows.append({
    "model": EMB_NAME_qwen3_emb_4b, "mode": "extrinsic", "threshold": ext_threshold_qwen3_emb_4b,
    "precision": ext_cv_rec_qwen3_emb_4b["precision"], "recall": ext_cv_rec_qwen3_emb_4b["recall"],
    "f1": ext_cv_rec_qwen3_emb_4b["f1"], "tpr": ext_row_qwen3_emb_4b["tpr"], "fpr": ext_row_qwen3_emb_4b["fpr"],
    "n_positive": int(extr_labels.sum()), "n_negative": int((extr_labels == 0).sum()),
})

# Step 4: Test on test_samples
results_path = os.path.join(save_dir, "results_extrinsic_similarity.jsonl")
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
print(f"Applying threshold {ext_threshold_qwen3_emb_4b:.4f} ({EMB_NAME_qwen3_emb_4b}) to test samples:")
for source, cve_id, ap_id, domain_pddl in test_samples:
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"  [{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    t0 = time.time()
    E_test = emb_model_qwen3_emb_4b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
    ref_texts = [ap["domain"] for ap in ref_aps]
    E_ref = emb_model_qwen3_emb_4b.encode(ref_texts, convert_to_tensor=True, normalize_embeddings=True)
    sims = (E_test @ E_ref.T).float().cpu().numpy()[0]
    elapsed = time.time() - t0
    best_sim = float(sims.max())
    best_js = [j for j in range(len(sims)) if float(sims[j]) == best_sim]
    pred = True if best_sim >= ext_threshold_qwen3_emb_4b else False
    for j, ref_ap in enumerate(ref_aps):
        sim = float(sims[j])
        p = True if sim >= ext_threshold_qwen3_emb_4b else False
        print(f"  [{source}] {cve_id}/{ap_id} vs {ref_ap['ap_id']}  sim={sim:.4f}  pred={p}  {elapsed:.3f}s")
    save_extrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_4b, ext_threshold_qwen3_emb_4b, cve_id, ap_id,
        [ref_aps[j]["ap_id"] for j in best_js], best_sim, pred, len(ref_aps), elapsed)
    print(f"  [{source}] {cve_id}/{ap_id} BEST={[ref_aps[j]['ap_id'] for j in best_js]}  sim={best_sim:.4f}  pred={pred}")

# Free model + reclaim GPU/CPU memory for next model
del emb_model_qwen3_emb_4b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')


In [ ]:
# === 3.3.2.1a Embedding Model: Qwen/Qwen3-Embedding-8B ===
# Reload if intrinsic section already freed the model
# Default settings (no max_seq truncation). Requires GPU >= 24GB; will OOM on smaller cards. Run on server.
if "emb_model_qwen3_emb_8b" not in dir():
    emb_model_qwen3_emb_8b = load_embedding_model("Qwen/Qwen3-Embedding-8B")
EMB_NAME_qwen3_emb_8b = "Qwen/Qwen3-Embedding-8B"

# Step 1: Compute scores
t_build = time.time()
_E = emb_model_qwen3_emb_8b.encode(extr_domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True, batch_size=8)
_sim_matrix = (_E @ _E.T).float().cpu().numpy()
del _E
n_ext = len(extr_domains)
ext_scores_qwen3_emb_8b = np.array([float(_sim_matrix[i, j]) for i in range(n_ext) for j in range(i+1, n_ext)])
del _sim_matrix
ext_build_seconds_qwen3_emb_8b = round(time.time() - t_build, 2)
print(f"{EMB_NAME_qwen3_emb_8b}: build_pairs {ext_build_seconds_qwen3_emb_8b}s")

# Step 2: CV calibration
t_cv = time.time()
ext_threshold_qwen3_emb_8b, ext_fold_thr_qwen3_emb_8b, ext_cv_pred_qwen3_emb_8b, ext_cv_true_qwen3_emb_8b = run_calibration(
    ext_scores_qwen3_emb_8b, extr_labels, extr_groups)
ext_cv_seconds_qwen3_emb_8b = round(time.time() - t_cv, 2)
print(f"  CV calibration {ext_cv_seconds_qwen3_emb_8b}s, threshold={ext_threshold_qwen3_emb_8b:.4f}")

# Step 3: Performance report + save CV
ext_row_qwen3_emb_8b = report_row(ext_cv_true_qwen3_emb_8b, ext_cv_pred_qwen3_emb_8b,
    metric="embedding", model=EMB_NAME_qwen3_emb_8b, mode="extrinsic", threshold=ext_threshold_qwen3_emb_8b)
print(f"  TPR={ext_row_qwen3_emb_8b['tpr']:.4f}, FPR={ext_row_qwen3_emb_8b['fpr']:.4f}, F1={ext_row_qwen3_emb_8b.get('1__f1-score', 0):.4f}")
print(classification_report(ext_cv_true_qwen3_emb_8b, ext_cv_pred_qwen3_emb_8b, zero_division=0))

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")
ext_cv_rec_qwen3_emb_8b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_8b, ext_threshold_qwen3_emb_8b, ext_fold_thr_qwen3_emb_8b,
    ext_row_qwen3_emb_8b, extr_labels, ext_build_seconds_qwen3_emb_8b, ext_cv_seconds_qwen3_emb_8b)

calibration_embedding_rows.append({
    "model": EMB_NAME_qwen3_emb_8b, "mode": "extrinsic", "threshold": ext_threshold_qwen3_emb_8b,
    "precision": ext_cv_rec_qwen3_emb_8b["precision"], "recall": ext_cv_rec_qwen3_emb_8b["recall"],
    "f1": ext_cv_rec_qwen3_emb_8b["f1"], "tpr": ext_row_qwen3_emb_8b["tpr"], "fpr": ext_row_qwen3_emb_8b["fpr"],
    "n_positive": int(extr_labels.sum()), "n_negative": int((extr_labels == 0).sum()),
})

# Step 4: Test on test_samples
results_path = os.path.join(save_dir, "results_extrinsic_similarity.jsonl")
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
print(f"Applying threshold {ext_threshold_qwen3_emb_8b:.4f} ({EMB_NAME_qwen3_emb_8b}) to test samples:")
for source, cve_id, ap_id, domain_pddl in test_samples:
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"  [{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    t0 = time.time()
    E_test = emb_model_qwen3_emb_8b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
    ref_texts = [ap["domain"] for ap in ref_aps]
    E_ref = emb_model_qwen3_emb_8b.encode(ref_texts, convert_to_tensor=True, normalize_embeddings=True)
    sims = (E_test @ E_ref.T).float().cpu().numpy()[0]
    elapsed = time.time() - t0
    best_sim = float(sims.max())
    best_js = [j for j in range(len(sims)) if float(sims[j]) == best_sim]
    pred = True if best_sim >= ext_threshold_qwen3_emb_8b else False
    for j, ref_ap in enumerate(ref_aps):
        sim = float(sims[j])
        p = True if sim >= ext_threshold_qwen3_emb_8b else False
        print(f"  [{source}] {cve_id}/{ap_id} vs {ref_ap['ap_id']}  sim={sim:.4f}  pred={p}  {elapsed:.3f}s")
    save_extrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_8b, ext_threshold_qwen3_emb_8b, cve_id, ap_id,
        [ref_aps[j]["ap_id"] for j in best_js], best_sim, pred, len(ref_aps), elapsed)
    print(f"  [{source}] {cve_id}/{ap_id} BEST={[ref_aps[j]['ap_id'] for j in best_js]}  sim={best_sim:.4f}  pred={pred}")

# Free model + reclaim GPU/CPU memory for next model
del emb_model_qwen3_emb_8b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')


##### 3.3.2.2 LLM as an Expert, Reference PDDL vs Candidate PDDL Match

##### 3.3.2.2a Binary Extrinsic (True/False)
`llm_eval_extrinsic_binary`: binary classification using `completion.md.jinja` (binary=True, extrinsic=True)


In [ ]:
def llm_eval_extrinsic_binary(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """Binary True/False: does the candidate match the CVE and reference?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=512,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


Block 2: call the function with the code and the specifications from the data set

In [ ]:
def preview_extrinsic_binary(n_calibration=2):
    sample = few_shot_pool[0]
    cal_data = load_calibration_data(CALIBRATION_DATA_PATH, 'extrinsic', exclude_cve=sample.cve_id)[:n_calibration]
    render_args = dict(
        binary=True, extrinsic=True,
        calibration_data=cal_data,
        cve_id=sample.cve_id, description=cve_descriptions.get(sample.cve_id, ''), domain=sample.domain_pddl,
    )
    ref = few_shot_pool[1] if len(few_shot_pool) > 1 else few_shot_pool[0]
    render_args['domain_reference'] = ref.domain_pddl
    prompt = eval_template.render(**render_args)
    print(f'--- binary extrinsic | {len(cal_data)} calibration examples | {len(prompt)} chars ---')
    print(prompt[:3000])
    if len(prompt) > 3000:
        print(f'\n... [{{len(prompt) - 3000}} chars truncated]')

interact(preview_extrinsic_binary, n_calibration=IntSlider(min=0, max=5, value=0, description='# Cal'));


interactive(children=(IntSlider(value=0, description='# Cal', max=5), Output()), _dom_classes=('widget-interac…

In [ ]:
t_start_ext_bin = time.time()
total_tokens_ext_bin = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
PRICE_PER_1K_IN, PRICE_PER_1K_OUT = 0.0004, 0.0004  # llama-3.3-70b-instruct $/1K tokens
results_path = os.path.join(save_dir, "results_extrinsic_binary.jsonl")

open(results_path, "w").close()

# ── Test on test_samples (1 reference + 1 bad) ──
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"[{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    t_domain = time.time()
    per_ref_labels = []
    total_cost = 0.0
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ref_ap in ref_aps:
        response, usage = llm_eval_extrinsic_binary(cve_id, description, ref_ap["domain"], domain_pddl, nvidia, NVIDIA_MODEL, seed=SEED)
        try:
            text = response.strip()
            if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
            raw_label = json.loads(text).get("label", "?")
            label = True if str(raw_label).lower() in ("true", "1", "yes") else False
        except Exception:
            label = None
        cost = round((usage.get("prompt_tokens", 0) * PRICE_PER_1K_IN + usage.get("completion_tokens", 0) * PRICE_PER_1K_OUT) / 1000, 6)
        total_cost += cost
        for k in total_usage: total_usage[k] += usage.get(k, 0)
        per_ref_labels.append(label)
        print(f"[{source}] {cve_id}/{ap_id:20s} vs {ref_ap['ap_id']} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")

    # Aggregated verdict: True if any reference gives True
    final_label = True if any(l is True for l in per_ref_labels) else False
    save_extrinsic_binary_result(results_path, NVIDIA_MODEL, SEED, 0.0, source, cve_id, ap_id, final_label, len(per_ref_labels), None not in per_ref_labels, total_usage, round(time.time() - t_domain, 2), round(total_cost, 6))
    print(f"[{source}] {cve_id}/{ap_id:20s} AGGREGATED | {final_label} ({sum(l=='True' for l in per_ref_labels)}/{len(per_ref_labels)} True)")


[reference] CVE-2022-40149/AP2                  vs AP1 | False | 2.27s  in=7676 out=10
[reference] CVE-2022-40149/AP2                  vs AP2 | True | 1.22s  in=7921 out=10
[reference] CVE-2022-40149/AP2                  AGGREGATED | True (0/2 True)
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 vs AP1 | False | 1.53s  in=7496 out=10
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 AGGREGATED | False (0/1 True)


gpt-4.1-mini extrinsic binary

In [ ]:

t_start_ext_bin_gpt = time.time()
total_tokens_ext_bin_gpt = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_extrinsic_binary.jsonl")

# ── Test on test_samples (1 reference + 1 bad) ──
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"[{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    t_domain = time.time()
    per_ref_labels = []
    total_cost = 0.0
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ref_ap in ref_aps:
        response, usage = llm_eval_extrinsic_binary(cve_id, description, ref_ap["domain"], domain_pddl, openai_client, GPT_MODEL, seed=SEED)
        try:
            text = response.strip()
            if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
            raw_label = json.loads(text).get("label", "?")
            label = True if str(raw_label).lower() in ("true", "1", "yes") else False
        except Exception:
            label = None
        cost = round((usage.get("prompt_tokens", 0) * PRICE_PER_1K_IN + usage.get("completion_tokens", 0) * PRICE_PER_1K_OUT) / 1000, 6)
        total_cost += cost
        for k in total_usage: total_usage[k] += usage.get(k, 0)
        per_ref_labels.append(label)
        print(f"[{source}] {cve_id}/{ap_id:20s} vs {ref_ap['ap_id']} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")

    # Aggregated verdict: True if any reference gives True
    final_label = True if any(l is True for l in per_ref_labels) else False
    save_extrinsic_binary_result(results_path, GPT_MODEL, SEED, 0.0, source, cve_id, ap_id, final_label, len(per_ref_labels), None not in per_ref_labels, total_usage, round(time.time() - t_domain, 2), round(total_cost, 6))
    print(f"[{source}] {cve_id}/{ap_id:20s} AGGREGATED | {final_label} ({sum(l=='True' for l in per_ref_labels)}/{len(per_ref_labels)} True)")


[reference] CVE-2022-40149/AP2                  vs AP1 | True | 2.62s  in=8229 out=9
[reference] CVE-2022-40149/AP2                  vs AP2 | True | 1.63s  in=8489 out=9
[reference] CVE-2022-40149/AP2                  AGGREGATED | True (0/2 True)
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 vs AP1 | True | 1.32s  in=8080 out=9
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 AGGREGATED | True (0/1 True)


##### 3.3.2.2b Scored Extrinsic (16-criteria, integer 0-5)
`llm_eval_extrinsic_scored`: 16-criteria scored evaluation (12 quality + 4 alignment) using `completion.md.jinja` (binary=False, extrinsic=True)


In [ ]:
def llm_eval_extrinsic_scored(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """16-criteria scored evaluation (12 quality + 4 reference-comparison, integer 0-5)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [ ]:
def preview_extrinsic_scored(n_calibration=2):
    sample = few_shot_pool[0]
    cal_data = load_calibration_data(CALIBRATION_DATA_PATH, 'extrinsic', exclude_cve=sample.cve_id)[:n_calibration]
    render_args = dict(
        binary=False, extrinsic=True,
        calibration_data=cal_data,
        cve_id=sample.cve_id, description=cve_descriptions.get(sample.cve_id, ''), domain=sample.domain_pddl,
    )
    ref = few_shot_pool[1] if len(few_shot_pool) > 1 else few_shot_pool[0]
    render_args['domain_reference'] = ref.domain_pddl
    prompt = eval_template.render(**render_args)
    print(f'--- scored extrinsic | {len(cal_data)} calibration examples | {len(prompt)} chars ---')
    print(prompt[:3000])
    if len(prompt) > 3000:
        print(f'\n... [{{len(prompt) - 3000}} chars truncated]')

interact(preview_extrinsic_scored, n_calibration=IntSlider(min=0, max=5, value=0, description='# Cal'));


interactive(children=(IntSlider(value=0, description='# Cal', max=5), Output()), _dom_classes=('widget-interac…

Block 2: call the function with the code and the specifications from the data set

NVIDIA_MODEL

In [ ]:
t_start_ext_scored = time.time()
total_tokens_ext_scored = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
PRICE_PER_1K_IN, PRICE_PER_1K_OUT = 0.0004, 0.0004  # llama-3.3-70b-instruct $/1K tokens
results_path = os.path.join(save_dir, "results_extrinsic_scored.jsonl")

EXT_SCORED_CRITERIA = SCORED_CRITERIA + ["R1","R2","R3","R4"]

open(results_path, "w").close()

# ── Test on test_samples (1 reference + 1 bad) ──
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"[{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    t_domain = time.time()
    per_ref_qmins = []
    total_cost = 0.0
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ref_ap in ref_aps:
        response, usage = llm_eval_extrinsic_scored(cve_id, description, ref_ap["domain"], domain_pddl, nvidia, NVIDIA_MODEL, seed=SEED)
        scores = parse_scored_response(response) or {"parse_error": True}
        if "parse_error" not in scores:
            quality_min = domain_min_score(scores)
            alignment_min = domain_min_score(scores, criteria=["R1","R2","R3","R4"])
            qmin = min(quality_min, alignment_min) if quality_min is not None and alignment_min is not None else None
        else:
            qmin = None
        verdict = True if qmin is not None and qmin >= 3 else False
        cost = round((usage.get("prompt_tokens", 0) * PRICE_PER_1K_IN + usage.get("completion_tokens", 0) * PRICE_PER_1K_OUT) / 1000, 6)
        total_cost += cost
        for k in total_usage: total_usage[k] += usage.get(k, 0)
        per_ref_qmins.append(qmin)
        print(f"[{source}] {cve_id}/{ap_id:20s} vs {ref_ap['ap_id']} | {verdict}({qmin}) | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")

    # Aggregated verdict: True if any reference gives qmin >= 3 (take best match)
    valid_qmins = [q for q in per_ref_qmins if q is not None]
    final_qmin = max(valid_qmins) if valid_qmins else None
    final_verdict = True if final_qmin is not None and final_qmin >= 3 else False
    save_extrinsic_scored_result(results_path, NVIDIA_MODEL, SEED, 0.0, source, cve_id, ap_id, final_verdict, final_qmin, len(per_ref_qmins), all(q is not None for q in per_ref_qmins), total_usage, round(time.time() - t_domain, 2), round(total_cost, 6))
    print(f"[{source}] {cve_id}/{ap_id:20s} AGGREGATED | {final_verdict}({final_qmin}) (best qmin={final_qmin} over {len(per_ref_qmins)} refs)")


[reference] CVE-2022-40149/AP2                  vs AP1 | True(4) | 9.51s  in=9331 out=135
[reference] CVE-2022-40149/AP2                  vs AP2 | True(5) | 4.09s  in=9576 out=135
[reference] CVE-2022-40149/AP2                  AGGREGATED | True(5) (best qmin=5 over 2 refs)
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 vs AP1 | False(2) | 12.63s  in=9151 out=135
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 AGGREGATED | False(2) (best qmin=2 over 1 refs)


gpt-4.1-mini

In [ ]:
t_start_ext_gpt = time.time()
total_tokens_ext_gpt = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_extrinsic_scored.jsonl")

# ── Test on test_samples (1 reference + 1 bad) ──
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"[{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    t_domain = time.time()
    per_ref_qmins = []
    total_cost = 0.0
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ref_ap in ref_aps:
        response, usage = llm_eval_extrinsic_scored(cve_id, description, ref_ap["domain"], domain_pddl, openai_client, GPT_MODEL, seed=SEED)
        scores = parse_scored_response(response) or {"parse_error": True}
        if "parse_error" not in scores:
            quality_min = domain_min_score(scores)
            alignment_min = domain_min_score(scores, criteria=["R1","R2","R3","R4"])
            qmin = min(quality_min, alignment_min) if quality_min is not None and alignment_min is not None else None
        else:
            qmin = None
        verdict = True if qmin is not None and qmin >= 3 else False
        cost = round((usage.get("prompt_tokens", 0) * PRICE_PER_1K_IN + usage.get("completion_tokens", 0) * PRICE_PER_1K_OUT) / 1000, 6)
        total_cost += cost
        for k in total_usage: total_usage[k] += usage.get(k, 0)
        per_ref_qmins.append(qmin)
        print(f"[{source}] {cve_id}/{ap_id:20s} vs {ref_ap['ap_id']} | {verdict}({qmin}) | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")

    # Aggregated verdict: True if any reference gives qmin >= 3 (take best match)
    valid_qmins = [q for q in per_ref_qmins if q is not None]
    final_qmin = max(valid_qmins) if valid_qmins else None
    final_verdict = True if final_qmin is not None and final_qmin >= 3 else False
    save_extrinsic_scored_result(results_path, GPT_MODEL, SEED, 0.0, source, cve_id, ap_id, final_verdict, final_qmin, len(per_ref_qmins), all(q is not None for q in per_ref_qmins), total_usage, round(time.time() - t_domain, 2), round(total_cost, 6))
    print(f"[{source}] {cve_id}/{ap_id:20s} AGGREGATED | {final_verdict}({final_qmin}) (best qmin={final_qmin} over {len(per_ref_qmins)} refs)")


[reference] CVE-2022-40149/AP2                  vs AP1 | True(5) | 11.62s  in=9894 out=670
[reference] CVE-2022-40149/AP2                  vs AP2 | True(5) | 3.17s  in=10154 out=134
[reference] CVE-2022-40149/AP2                  AGGREGATED | True(5) (best qmin=5 over 2 refs)
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 vs AP1 | True(5) | 18.23s  in=9745 out=998
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 AGGREGATED | True(5) (best qmin=5 over 1 refs)


## 4. Inter-Rater Agreement (Fleiss' Kappa)

Compute Fleiss' kappa among 14 raters



In [ ]:
def load_jsonl(fpath):
    if not os.path.exists(fpath):
        return []
    with open(fpath) as f:
        return [json.loads(line) for line in f if line.strip()]

# ── Load all results ──
results_dir_i = os.path.join(RESULTS_BASE, "semantic", "intrinsic")
results_dir_e = os.path.join(RESULTS_BASE, "semantic", "extrinsic")

syntax_data     = load_jsonl(os.path.join(RESULTS_BASE, "syntax", "results_syntax.jsonl"))
solv_data       = load_jsonl(os.path.join(RESULTS_BASE, "solvability", "results_solvability.jsonl"))
intr_sim_data   = load_jsonl(os.path.join(results_dir_i, "similarity", "results_intrinsic_similarity.jsonl"))
intr_bin_data   = load_jsonl(os.path.join(results_dir_i, "llm-as-experts", "results_intrinsic_binary.jsonl"))
intr_scored_data= load_jsonl(os.path.join(results_dir_i, "llm-as-experts", "results_intrinsic_scored.jsonl"))
extr_sim_data   = load_jsonl(os.path.join(results_dir_e, "similarity", "results_extrinsic_similarity.jsonl"))
extr_bin_data   = load_jsonl(os.path.join(results_dir_e, "llm-as-experts", "results_extrinsic_binary.jsonl"))
extr_scored_data= load_jsonl(os.path.join(results_dir_e, "llm-as-experts", "results_extrinsic_scored.jsonl"))

def to_binary(val):
    s = str(val).strip()
    if s.startswith("True") or s in ("true", "1", "yes"): return 1
    if s.startswith("False") or s in ("false", "0", "no"): return 0
    return None

# ── Build rating matrix ──
ratings = defaultdict(dict)

for d in syntax_data:
    k = (d.get("cve_id", ""), d.get("ap_id", ""))
    ratings[k]["syntax"] = to_binary(d.get("syntax_ok"))

for d in solv_data:
    k = (d.get("cve_id", ""), d.get("ap_id", ""))
    ratings[k]["solvability"] = to_binary(d.get("solvable"))

for d in intr_sim_data:
    k = (d.get("cve_id", ""), d.get("ap_id", ""))
    short = model_short_name(d.get("model", "unknown"))
    ratings[k][f"intr_sim_{short}"] = to_binary(d.get("prediction"))

for d in intr_bin_data:
    k = (d.get("cve_id", ""), d.get("ap_id", ""))
    short = model_short_name(d.get("llm_model", ""))
    ratings[k][f"intr_bin_{short}"] = to_binary(d.get("label"))

for d in intr_scored_data:
    k = (d.get("cve_id", ""), d.get("ap_id", ""))
    short = model_short_name(d.get("llm_model", ""))
    ratings[k][f"intr_scored_{short}"] = to_binary(d.get("verdict"))

# Extrinsic: one aggregated record per domain, read directly
for d in extr_sim_data:
    k = (d.get("cve_id", ""), d.get("test_ap", d.get("ap_id", "")))
    short = model_short_name(d.get("model", "unknown"))
    ratings[k][f"extr_sim_{short}"] = to_binary(d.get("prediction"))

for d in extr_bin_data:
    k = (d.get("cve_id", ""), d.get("generated", ""))
    short = model_short_name(d.get("llm_model", ""))
    ratings[k][f"extr_bin_{short}"] = to_binary(d.get("label"))

for d in extr_scored_data:
    k = (d.get("cve_id", ""), d.get("generated", ""))
    short = model_short_name(d.get("llm_model", ""))
    ratings[k][f"extr_scored_{short}"] = to_binary(d.get("verdict"))

# ── Determine all raters present ──
all_raters = sorted(set(r for v in ratings.values() for r in v))
print(f"Raters ({len(all_raters)}): {all_raters}")

# ── Build matrix: only domains with ALL raters ──
matrix = []
domain_keys_used = []
for k, r in sorted(ratings.items()):
    row = [r.get(rater) for rater in all_raters]
    if all(v is not None for v in row):
        matrix.append(row)
        domain_keys_used.append(k)

print(f"Domains with all raters: {len(matrix)} / {len(ratings)}")

def compute_kappa(ratings_dict, rater_filter=None, label=""):
    """Compute Fleiss' kappa for a subset of raters."""
    if rater_filter:
        filtered_raters = sorted([r for r in set(r for v in ratings_dict.values() for r in v) if any(f in r for f in rater_filter)])
    else:
        filtered_raters = sorted(set(r for v in ratings_dict.values() for r in v))
    
    if len(filtered_raters) < 2:
        print(f"  {label}: insufficient raters ({len(filtered_raters)})")
        return None
    
    matrix = []
    keys_used = []
    for k, r in sorted(ratings_dict.items()):
        row = [r.get(rater) for rater in filtered_raters]
        if all(v is not None for v in row):
            matrix.append(row)
            keys_used.append(k)
    
    if len(matrix) < 2:
        print(f"  {label}: insufficient subjects ({len(matrix)})")
        return None
    
    matrix_np = np.array(matrix)
    agg_table, _ = aggregate_raters(matrix_np)
    kappa = _fleiss_kappa(agg_table, method="fleiss")
    
    if kappa < 0: interp = "Poor"
    elif kappa < 0.20: interp = "Slight"
    elif kappa < 0.40: interp = "Fair"
    elif kappa < 0.60: interp = "Moderate"
    elif kappa < 0.80: interp = "Substantial"
    else: interp = "Almost Perfect"
    
    print(f"\n{label} Fleiss' Kappa = {kappa:.4f}  ({interp})")
    print(f"  Subjects: {len(matrix)}, Raters: {len(filtered_raters)}")
    print(f"  Raters: {filtered_raters}")
    
    print(f"  Per-domain ratings:")
    for i, k in enumerate(keys_used):
        row = matrix[i]
        true_count = sum(row)
        agree = "AGREE" if len(set(row)) == 1 else "DISAGREE"
        print(f"    {k[0]}/{k[1]:30s} | {true_count}/{len(row)} True | {agree}")
    
    return {
        "scope": label,
        "timestamp": datetime.now().isoformat(),
        "kappa": round(kappa, 4),
        "interpretation": interp,
        "n_subjects": len(matrix),
        "n_raters": len(filtered_raters),
        "raters": filtered_raters,
    }

# ── Compute 3 kappa values ──
kappa_results = []

# 1. Overall
result = compute_kappa(ratings, label="Overall")
if result: kappa_results.append(result)

# 2. Intrinsic semantic (syntax + solvability + intrinsic similarity + intrinsic LLM)
result = compute_kappa(ratings, rater_filter=["intr_sim_", "intr_bin_", "intr_scored_"], label="Intrinsic Semantic")
if result: kappa_results.append(result)

# 3. Extrinsic semantic (extrinsic similarity + extrinsic LLM)
result = compute_kappa(ratings, rater_filter=["extr_sim_", "extr_bin_", "extr_scored_"], label="Extrinsic Semantic")
if result: kappa_results.append(result)

# ── Save ──
kappa_path = os.path.join(RESULTS_BASE, "agreement", "fleiss_kappa.jsonl")
os.makedirs(os.path.dirname(kappa_path), exist_ok=True)
with open(kappa_path, "w") as f:
    for r in kappa_results:
        f.write(json.dumps(r) + "\n")
print(f"\nSaved {len(kappa_results)} kappa results")


Raters (14): ['extr_bin_gpt-4.1-mini', 'extr_bin_llama-3.3-70b', 'extr_scored_gpt-4.1-mini', 'extr_scored_llama-3.3-70b', 'extr_sim_all-MiniLM-L6-v2', 'extr_sim_bge-base-en-v1.5', 'intr_bin_gpt-4.1-mini', 'intr_bin_llama-3.3-70b', 'intr_scored_gpt-4.1-mini', 'intr_scored_llama-3.3-70b', 'intr_sim_all-MiniLM-L6-v2', 'intr_sim_bge-base-en-v1.5', 'solvability', 'syntax']
Domains with all raters: 2 / 2

Overall Fleiss' Kappa = -0.0676  (Poor)
  Subjects: 2, Raters: 14
  Raters: ['extr_bin_gpt-4.1-mini', 'extr_bin_llama-3.3-70b', 'extr_scored_gpt-4.1-mini', 'extr_scored_llama-3.3-70b', 'extr_sim_all-MiniLM-L6-v2', 'extr_sim_bge-base-en-v1.5', 'intr_bin_gpt-4.1-mini', 'intr_bin_llama-3.3-70b', 'intr_scored_gpt-4.1-mini', 'intr_scored_llama-3.3-70b', 'intr_sim_all-MiniLM-L6-v2', 'intr_sim_bge-base-en-v1.5', 'solvability', 'syntax']
  Per-domain ratings:
    CVE-2022-40149/AP2                            | 12/14 True | DISAGREE
    CVE-2024-38809/AP1_inject_capability_violation_rep1 | 11/14 Tru

## 5. Evaluation Summary
Aggregate all evaluation results into one row per domain.


In [ ]:
def load_jsonl(fpath):
    if not os.path.exists(fpath):
        return []
    with open(fpath) as f:
        return [json.loads(line) for line in f if line.strip()]

results_dir_i = os.path.join(RESULTS_BASE, "semantic", "intrinsic")
results_dir_e = os.path.join(RESULTS_BASE, "semantic", "extrinsic")

syntax_data      = load_jsonl(os.path.join(RESULTS_BASE, "syntax", "results_syntax.jsonl"))
solv_data        = load_jsonl(os.path.join(RESULTS_BASE, "solvability", "results_solvability.jsonl"))
intr_sim_data    = load_jsonl(os.path.join(results_dir_i, "similarity", "results_intrinsic_similarity.jsonl"))
intr_bin_data    = load_jsonl(os.path.join(results_dir_i, "llm-as-experts", "results_intrinsic_binary.jsonl"))
intr_scored_data = load_jsonl(os.path.join(results_dir_i, "llm-as-experts", "results_intrinsic_scored.jsonl"))
extr_sim_data    = load_jsonl(os.path.join(results_dir_e, "similarity", "results_extrinsic_similarity.jsonl"))
extr_bin_data    = load_jsonl(os.path.join(results_dir_e, "llm-as-experts", "results_extrinsic_binary.jsonl"))
extr_scored_data = load_jsonl(os.path.join(results_dir_e, "llm-as-experts", "results_extrinsic_scored.jsonl"))
kappa_data       = load_jsonl(os.path.join(RESULTS_BASE, "agreement", "fleiss_kappa.jsonl"))

# ── Build per-domain summary ──
summary = defaultdict(dict)

for d in syntax_data:
    k = (d.get("cve_id", ""), d.get("ap_id", ""))
    summary[k].update({"timestamp": d.get("timestamp", ""), "source": d.get("source", ""), "cve_id": d.get("cve_id", ""), "ap_id": d.get("ap_id", ""), "syntax": "True" if d.get("syntax_ok") else "False", "syntax_time": d.get("elapsed_seconds", 0)})

for d in solv_data:
    k = (d.get("cve_id", ""), d.get("ap_id", ""))
    summary[k].update({"solvability": "True" if d.get("solvable") else "False", "solvability_time": d.get("elapsed_seconds", 0)})

for d in intr_sim_data:
    k = (d.get("cve_id", ""), d.get("ap_id", ""))
    short = model_short_name(d.get("model", "unknown"))
    summary[k][f"intr_sim_{short}"] = d.get("prediction", "")
    summary[k][f"intr_sim_{short}_time"] = d.get("elapsed_seconds", 0)

for d in intr_bin_data:
    k = (d.get("cve_id", ""), d.get("ap_id", ""))
    short = model_short_name(d.get("llm_model", ""))
    summary[k][f"intr_bin_{short}"] = d.get("label", "")
    summary[k][f"intr_bin_{short}_time"] = d.get("usage", {}).get("elapsed_seconds", 0)
    summary[k][f"intr_bin_{short}_tokens"] = d.get("usage", {}).get("total_tokens", 0)
    summary[k][f"intr_bin_{short}_cost"] = d.get("cost_usd", 0)

for d in intr_scored_data:
    k = (d.get("cve_id", ""), d.get("ap_id", ""))
    short = model_short_name(d.get("llm_model", ""))
    summary[k][f"intr_scored_{short}"] = str(d.get("verdict", ""))
    summary[k][f"intr_scored_{short}_time"] = d.get("usage", {}).get("elapsed_seconds", 0)
    summary[k][f"intr_scored_{short}_tokens"] = d.get("usage", {}).get("total_tokens", 0)
    summary[k][f"intr_scored_{short}_cost"] = d.get("cost_usd", 0)

# Extrinsic: one aggregated record per domain, read directly
for d in extr_sim_data:
    k = (d.get("cve_id", ""), d.get("test_ap", d.get("ap_id", "")))
    short = model_short_name(d.get("model", "unknown"))
    summary[k][f"extr_sim_{short}"] = d.get("prediction", "")
    summary[k][f"extr_sim_{short}_time"] = d.get("elapsed_seconds", 0)

for d in extr_bin_data:
    k = (d.get("cve_id", ""), d.get("generated", ""))
    short = model_short_name(d.get("llm_model", ""))
    summary[k][f"extr_bin_{short}"] = d.get("label", "")
    summary[k][f"extr_bin_{short}_time"] = d.get("usage", {}).get("elapsed_seconds", 0)
    summary[k][f"extr_bin_{short}_tokens"] = d.get("usage", {}).get("total_tokens", 0)
    summary[k][f"extr_bin_{short}_cost"] = d.get("cost_usd", 0)

for d in extr_scored_data:
    k = (d.get("cve_id", ""), d.get("generated", ""))
    short = model_short_name(d.get("llm_model", ""))
    summary[k][f"extr_scored_{short}"] = str(d.get("verdict", ""))
    summary[k][f"extr_scored_{short}_time"] = d.get("usage", {}).get("elapsed_seconds", 0)
    summary[k][f"extr_scored_{short}_tokens"] = d.get("usage", {}).get("total_tokens", 0)
    summary[k][f"extr_scored_{short}_cost"] = d.get("cost_usd", 0)

# ── Discover all column names dynamically ──
all_cols = set()
for v in summary.values():
    all_cols.update(v.keys())

_eval_raw = [c for c in all_cols if not c.endswith(("_time", "_tokens", "_cost", "_min"))
             and c not in ("timestamp", "source", "cve_id", "ap_id")]

_order_prefix = ["syntax", "solvability", "intr_sim_", "intr_bin_", "intr_scored_", "extr_sim_", "extr_bin_", "extr_scored_"]

def _col_sort_key(col):
    for i, prefix in enumerate(_order_prefix):
        if col == prefix.rstrip("_") or col.startswith(prefix):
            return (i, col)
    return (len(_order_prefix), col)

eval_cols  = sorted(_eval_raw, key=_col_sort_key)
time_cols  = sorted([c for c in all_cols if c.endswith("_time")])
token_cols = sorted([c for c in all_cols if c.endswith("_tokens")])
cost_cols  = sorted([c for c in all_cols if c.endswith("_cost")])

# ── Build DataFrame ──
rows = []
for k, v in sorted(summary.items()):
    row = {"timestamp": v.get("timestamp", ""), "domain": f"{v.get('cve_id', k[0])}/{v.get('ap_id', k[1])}", "source": v.get("source", "")}
    for c in eval_cols:
        row[c] = v.get(c, "")
    row["total_time_s"]   = round(sum(v.get(c, 0) or 0 for c in time_cols  if isinstance(v.get(c), (int, float))), 2)
    row["total_tokens"]   = sum(v.get(c, 0) or 0 for c in token_cols if isinstance(v.get(c), (int, float)))
    row["total_cost_usd"] = round(sum(v.get(c, 0) or 0 for c in cost_cols  if isinstance(v.get(c), (int, float))), 6)
    rows.append(row)

df = pd.DataFrame(rows)

# ── Print ──
print("=" * 120)
print("EVALUATION SUMMARY")
print("=" * 120)
print(df.to_string(index=False))

print("\n--- Fleiss' Kappa (inter-rater agreement) ---")
if kappa_data:
    for kd in kappa_data:
        print(f"  [{kd.get('scope', 'Overall')}] Kappa = {kd.get('kappa', 'N/A')}  ({kd.get('interpretation', '')})")
        print(f"    Subjects: {kd.get('n_subjects', '')}, Raters: {kd.get('n_raters', '')}")
else:
    print("  No kappa data found")

print(f"\n--- Totals ---")
print(f"  Domains evaluated: {len(df)}")
print(f"  Total time: {df['total_time_s'].sum():.2f}s")
print(f"  Total tokens: {int(df['total_tokens'].sum())}")
print(f"  Total cost: ${df['total_cost_usd'].sum():.4f}")

# ── Save ──
summary_path = os.path.join(RESULTS_BASE, "results_summary.jsonl")
with open(summary_path, "w") as f:
    for _, row in df.iterrows():
        f.write(json.dumps(row.to_dict()) + "\n")


EVALUATION SUMMARY
                 timestamp                                              domain    source syntax solvability  intr_sim_all-MiniLM-L6-v2  intr_sim_bge-base-en-v1.5  intr_bin_gpt-4.1-mini  intr_bin_llama-3.3-70b intr_scored_gpt-4.1-mini intr_scored_llama-3.3-70b  extr_sim_all-MiniLM-L6-v2  extr_sim_bge-base-en-v1.5  extr_bin_gpt-4.1-mini  extr_bin_llama-3.3-70b extr_scored_gpt-4.1-mini extr_scored_llama-3.3-70b  total_time_s  total_tokens  total_cost_usd
2026-06-01T19:02:05.638407                                  CVE-2022-40149/AP2 reference   True        True                      False                      False                   True                    True                     True                      True                       True                       True                   True                    True                     True                      True         91.85         93860               0
2026-06-01T19:02:35.690655 CVE-2024-38809/AP1_inject_capability_viola